# Phase 6 — Abstractive Legal Summarisation with the pre-trained **Mamba** (decoder-only, recurrent)

Mamba is a causal language model with no attention, so it has no cross-attention and no encoder, and neither is added. As with GQLA, the document is placed *before* the summary in one sequence. Mamba's recurrent state carries what it read into the summary tokens. The document is first compressed into a short memory:

```
document → 512-token chunks (K spread over the judgment)
        → frozen Phase-1 Mamba, run once     → hidden states of each chunk
        → m segment-means per chunk          → K·m memory slots   (cached fp16 on Drive)
        → MemoryBridge (trainable): chunk/slot positions + 1 bidirectional Transformer layer
                                   + projection into the token-embedding space
        → Mamba (adapted with the chosen method), one left-to-right pass:

          [ pad … pad | prompt? | mem_1 … mem_N | <s> | y_1 … y_L | </s> ]
                                                  └── loss only here (label smoothing 0.1)
```

**Mamba compared with GQLA:** there is **no 512-token window**. Generation is a recurrent step with constant memory per token, so summaries are not squeezed into what the memory leaves of 512 positions; the budget is set by the reference lengths (capped at 768). `mem_mode="text"` drops the memory and feeds 2048 raw document tokens directly, an ablation only a recurrent model can run cheaply.

**No re-pre-training is needed.** The only new weights are the bridge (~3M parameters) plus whatever the adaptation method adds.

**Methods** (`method=`): `full` · `partial` · `lora` · `lorma` (LoRMA⁺, compulsory) · `qlora` (NF4 4-bit base + LoRA) · `adapter` (bottleneck after every mixer) · `prompt` · `frozen` (bridge only) · `kd` (10-block student initialised from the fine-tuned teacher, CE + temperature-scaled KL). LoRA/LoRMA/QLoRA adapt `in_proj`, `x_proj` and `out_proj`; the SSM parameters (`A_log`, `D`, `dt_proj`, `conv1d`) stay frozen, so the fused scan kernel is unchanged. Prefix tuning needs attention keys and values, so it does not apply; prompt tuning is its counterpart.

**What is new relative to the Phase-5 Mamba notebook**
* **Left-padding mask:** the `in_proj` and conv outputs are zeroed at padding, so the SSM state stays exactly zero through padding and batched prefixes behave like unpadded ones.
* **Recurrent decoding:** `mixer_step` updates the conv window and SSM state one token at a time, and the prefix pass returns those states (via `return_last_state` on the fused kernel).
* The LM head is kept, because it writes the summary.

**Run order:** top to bottom. Cell 1b checks four things against the Phase-1 maths: the patched forward, left padding, recurrent steps against the full forward, and the fused kernel against the pure-PyTorch scan (it switches the kernel off if they disagree). The config cell checks that every PEFT module is the identity at initialisation. Then run the smoke test, then the experiments.

In [ ]:
# ============================================================
# OPTIONAL (GPU runtime): fused Mamba CUDA kernels
#   - mamba-ssm      -> fused selective scan (the big one)
#   - causal-conv1d  -> fused causal conv; supports kernel width 2-4 ONLY. Your Phase-1 model
#                       has d_conv=16, so this notebook cannot use it (the conv runs through cuDNN).
# Pre-built wheels exist only for some torch/CUDA/python combinations; otherwise pip compiles from
# source (can take 10-30 min on Colab). The notebook works without it (slower pure-PyTorch scan).
# Restart the runtime after installing, then run everything from the top.
# ============================================================
# !pip install -q ninja packaging
# !pip install -q mamba-ssm --no-build-isolation
!pip -q install rouge-score bert-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import re
import json
import heapq
import collections

try:
    import psutil
    _PROC = psutil.Process(os.getpid())
except Exception:                       # psutil optional
    _PROC = None


def _ram_gb():
    return _PROC.memory_info().rss / (1024 ** 3) if _PROC else 0.0


# ============================================================
# MUST BE IDENTICAL FOR TRAINING AND INFERENCE
# ============================================================
WORD_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)


# ---- multiprocessing helpers (module level so they can be pickled) ----
_WORKER = None

def _init_worker(tok):
    global _WORKER
    _WORKER = tok

def _encode_chunk(args):
    texts, add_special = args
    return [_WORKER.encode(t, add_special) for t in texts]


class BPETokenizer:

    _HEAP_THRESHOLD = 24        # words longer than this use the heap merger
    _CACHE_LIMIT = 500_000

    # ========================================================
    # INIT
    # ========================================================
    def __init__(self, vocab_size, end_marker="</w>", reserved_tokens=None):
        self.vocab_size = vocab_size
        self.end_marker = end_marker
        self.reserved_tokens = (
            reserved_tokens
            if reserved_tokens is not None
            else ["<pad>", "<unk>", "<s>", "</s>"]
        )

        self.itos = list(self.reserved_tokens)
        self.stoi = {t: i for i, t in enumerate(self.reserved_tokens)}

        # (id_a, id_b) -> merge rank. Rank r created token first_merge_id + r.
        self.merges = {}

        self._ready = False
        self._cache = {}

    # ========================================================
    # SAVE / LOAD
    # ========================================================
    def save(self, path):
        data = {
            "vocab_size": self.vocab_size,
            "end_marker": self.end_marker,
            "reserved_tokens": self.reserved_tokens,
            "itos": self.itos,
            "stoi": self.stoi,
            "merges": [[list(p), r] for p, r in self.merges.items()],
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)   # no indent: much smaller/faster

        print(f"Tokenizer saved to: {path}")
        print(f"Vocabulary size: {len(self.itos):,}")
        print(f"Number of merges: {len(self.merges):,}")

    @staticmethod
    def from_json(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        tok = BPETokenizer(
            vocab_size=data["vocab_size"],
            end_marker=data.get("end_marker", "</w>"),
            reserved_tokens=data.get(
                "reserved_tokens", ["<pad>", "<unk>", "<s>", "</s>"]
            ),
        )
        tok.itos = data["itos"]
        tok.stoi = data["stoi"]
        tok.merges = {tuple(p): r for p, r in data["merges"]}
        tok._ready = False
        tok._cache = {}
        return tok

    # ========================================================
    # BASE VOCAB
    # ========================================================
    def _initialize_bytes(self):
        """Reserved tokens, then 256 byte tokens, then </w>."""
        for b in range(256):
            tok = bytes([b]).decode("latin-1")
            if tok not in self.stoi:
                self.stoi[tok] = len(self.itos)
                self.itos.append(tok)

        if self.end_marker not in self.stoi:
            self.stoi[self.end_marker] = len(self.itos)
            self.itos.append(self.end_marker)

    # ========================================================
    # WORD FREQUENCIES  (memory-safe)
    # ========================================================
    def _get_initial_vocab(
        self,
        corpus_texts,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        """
        Count words as plain strings (far lighter than tuples of ints),
        periodically dropping the long tail (typos, ids, citation numbers),
        with a hard cap on the number of unique words. Only the survivors are
        converted to byte-id tuples.

        Dropped rare words are still encodable at inference through byte
        fallback, so this costs almost nothing in tokenizer quality.
        """
        self._initialize_bytes()
        end_id = self.stoi[self.end_marker]

        counts = collections.Counter()
        threshold = prune_threshold
        n = 0
        last_ram = _ram_gb()

        def prune(th):
            return collections.Counter(
                {w: c for w, c in counts.items() if c >= th}
            )

        for n, text in enumerate(corpus_texts, 1):
            if not isinstance(text, str):
                continue

            if len(text) > max_doc_chars:
                print(
                    f"  [warn] example {n:,} has {len(text):,} chars "
                    f"-> truncated to {max_doc_chars:,}"
                )
                text = text[:max_doc_chars]

            # generator: never builds a giant list of words in memory
            counts.update(m.group() for m in WORD_RE.finditer(text))

            if n % 100 == 0:
                ram = _ram_gb()
                if ram - last_ram > 0.3:
                    print(f"  [warn] RAM jumped {last_ram:.2f} -> {ram:.2f} GB near example {n:,}")
                last_ram = ram

            if n % prune_every == 0:
                counts = prune(threshold)

            # hard safety cap: raise threshold until we fit
            while len(counts) > max_unique_words:
                threshold += 1
                counts = prune(threshold)
                print(f"  cap hit -> prune threshold raised to {threshold}")

            if n % 1000 == 0:
                print(
                    f"Examples: {n:,} | Unique words: {len(counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if n >= max_examples:
                break

        counts = prune(prune_threshold)     # final prune

        # convert survivors to id tuples
        byte_ids = [self.stoi[bytes([b]).decode("latin-1")] for b in range(256)]
        word_freqs = {}
        for w, c in counts.items():
            ids = tuple(byte_ids[b] for b in w.encode("utf-8")) + (end_id,)
            word_freqs[ids] = word_freqs.get(ids, 0) + c
        del counts

        print(f"\nFinished reading corpus. Examples: {n:,}")
        print(f"Unique words kept: {len(word_freqs):,}")
        return word_freqs

    # ========================================================
    # TRAIN
    # ========================================================
    def train(
        self,
        corpus_texts,
        save_checkpoints=None,
        checkpoint_dir=None,
        max_examples=200_000,
        prune_every=5_000,
        prune_threshold=2,
        max_unique_words=2_000_000,
        max_doc_chars=500_000,
    ):
        save_checkpoints = set(save_checkpoints or [])
        checkpoint_dir = checkpoint_dir or "."
        os.makedirs(checkpoint_dir, exist_ok=True)

        # ---- Step 1: word frequencies --------------------------------
        print("\nSTEP 1: word frequencies")
        word_freqs = self._get_initial_vocab(
            corpus_texts,
            max_examples=max_examples,
            prune_every=prune_every,
            prune_threshold=prune_threshold,
            max_unique_words=max_unique_words,
            max_doc_chars=max_doc_chars,
        )

        words = [list(w) for w in word_freqs]
        freqs = list(word_freqs.values())
        del word_freqs

        # ---- Step 2: pair statistics ---------------------------------
        print("\nSTEP 2: pair statistics")
        pair_counts = collections.defaultdict(int)
        pair_to_words = collections.defaultdict(set)     # pair -> word indices

        for idx, (w, f) in enumerate(zip(words, freqs)):
            for pair in zip(w, w[1:]):
                pair_counts[pair] += f
                pair_to_words[pair].add(idx)

        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        print(f"Initial vocab size: {len(self.itos):,}")
        print(f"Unique words:       {len(words):,}")
        print(f"Unique pairs:       {len(pair_counts):,}")
        print(f"RAM: {_ram_gb():.2f} GB")

        # ---- Step 3: merges ------------------------------------------
        print("\nSTEP 3: BPE merges")

        while len(self.itos) < self.vocab_size:

            # pop until an up-to-date entry is found (lazy deletion)
            best_pair, best_freq = None, 0
            while heap:
                neg, pair = heapq.heappop(heap)
                if -neg > 0 and pair_counts.get(pair, 0) == -neg:
                    best_pair, best_freq = pair, -neg
                    break

            if best_pair is None:
                print("No more pairs available.")
                break

            a, b = best_pair

            new_id = len(self.itos)
            new_str = self.itos[a] + self.itos[b]
            self.itos.append(new_str)
            self.stoi.setdefault(new_str, new_id)
            self.merges[best_pair] = len(self.merges)

            affected = pair_to_words.pop(best_pair, set())
            changed = set()

            for idx in affected:
                w, f = words[idx], freqs[idx]

                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                    if p != best_pair:
                        s = pair_to_words.get(p)
                        if s is not None:
                            s.discard(idx)

                out, i, n = [], 0, len(w)
                while i < n:
                    if i < n - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                    else:
                        out.append(w[i])
                        i += 1
                words[idx] = out

                for p in zip(out, out[1:]):
                    pair_counts[p] += f
                    pair_to_words[p].add(idx)
                    changed.add(p)

            pair_counts.pop(best_pair, None)
            changed.discard(best_pair)

            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_to_words.pop(p, None)

            # rebuild heap if it is mostly stale (bounds heap memory)
            if len(heap) > 4 * len(pair_counts) + 10_000:
                heap = [(-c, p) for p, c in pair_counts.items()]
                heapq.heapify(heap)

            vocab_now = len(self.itos)

            if vocab_now % 1000 == 0:
                print(
                    f"Vocab: {vocab_now:,} | merge freq: {best_freq:,} | "
                    f"affected: {len(affected):,} | pairs: {len(pair_counts):,} "
                    f"| RAM: {_ram_gb():.2f} GB"
                )

            if vocab_now in save_checkpoints:
                self.save(os.path.join(checkpoint_dir, f"bpe_{vocab_now}.json"))

        self._ready = False
        self._cache = {}

        print("\nBPE training complete.")
        print(f"Final vocabulary size: {len(self.itos):,}")
        print(f"Merges learned:        {len(self.merges):,}")

    # ========================================================
    # INFERENCE SETUP
    # ========================================================
    def _prepare_inference(self):
        if self._ready:
            return

        self._byte_ids = [
            self.stoi[bytes([b]).decode("latin-1")] for b in range(256)
        ]
        self._first_merge_id = len(self.itos) - len(self.merges)
        self._end_id = self.stoi.get(self.end_marker, self.stoi["<unk>"])
        self._bos = self.stoi["<s>"]
        self._eos = self.stoi["</s>"]

        # sanity check: rank r must have created token first_merge_id + r
        base = self._first_merge_id
        for (a, b), r in self.merges.items():
            if self.itos[base + r] != self.itos[a] + self.itos[b]:
                raise ValueError(
                    "Merge/vocab mismatch: cannot infer merge ids from ranks."
                )

        # integer keys: faster than tuple keys
        self._M = len(self.itos)
        M = self._M
        self._ranks = {a * M + b: r for (a, b), r in self.merges.items()}

        self._cache = {}
        self._ready = True

    # ========================================================
    # MERGE STRATEGIES
    # ========================================================
    def _merge_small(self, ids):
        """Repeated best-rank scan. Fastest for short words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        while len(ids) > 1:
            best, best_i = None, -1
            for i in range(len(ids) - 1):
                r = get(ids[i] * M + ids[i + 1])
                if r is not None and (best is None or r < best):
                    best, best_i = r, i

            if best is None:
                break

            a, b, new = ids[best_i], ids[best_i + 1], base + best
            out, i, n = [], 0, len(ids)
            while i < n:
                if i < n - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out

        return ids

    def _merge_heap(self, ids):
        """Linked list + min-heap: O(n log n) for long words."""
        get, M, base = self._ranks.get, self._M, self._first_merge_id

        n = len(ids)
        nxt = list(range(1, n + 1))
        nxt[-1] = -1
        prv = list(range(-1, n - 1))
        alive = [True] * n

        heap = []
        for i in range(n - 1):
            r = get(ids[i] * M + ids[i + 1])
            if r is not None:
                heap.append((r, i))
        heapq.heapify(heap)

        while heap:
            r, i = heapq.heappop(heap)
            if not alive[i]:
                continue
            j = nxt[i]
            if j == -1 or get(ids[i] * M + ids[j]) != r:
                continue                        # stale entry

            ids[i] = base + r
            alive[j] = False
            k = nxt[j]
            nxt[i] = k
            if k != -1:
                prv[k] = i

            p = prv[i]
            if p != -1:
                r2 = get(ids[p] * M + ids[i])
                if r2 is not None:
                    heapq.heappush(heap, (r2, p))
            if k != -1:
                r2 = get(ids[i] * M + ids[k])
                if r2 is not None:
                    heapq.heappush(heap, (r2, i))

        return [ids[i] for i in range(n) if alive[i]]

    def _bpe_word(self, word):
        hit = self._cache.get(word)
        if hit is not None:
            return hit

        ids = [self._byte_ids[b] for b in word.encode("utf-8")]
        ids.append(self._end_id)

        if len(ids) <= self._HEAP_THRESHOLD:
            ids = self._merge_small(ids)
        else:
            ids = self._merge_heap(ids)

        result = tuple(ids)
        if len(self._cache) < self._CACHE_LIMIT:
            self._cache[word] = result
        return result

    # ========================================================
    # PUBLIC API
    # ========================================================
    def encode(self, text, add_special_tokens=True):
        """Fast path: token ids only."""
        self._prepare_inference()

        out = [self._bos] if add_special_tokens else []
        extend = out.extend
        cache_get = self._cache.get
        bpe = self._bpe_word

        for w in WORD_RE.findall(text):
            t = cache_get(w)
            if t is None:
                t = bpe(w)
            extend(t)

        if add_special_tokens:
            out.append(self._eos)
        return out

    def tokenize(self, text, add_special_tokens=True):
        """Returns (token_ids, token_strings). Use encode() for bulk work."""
        ids = self.encode(text, add_special_tokens)
        itos = self.itos
        return ids, [itos[i] for i in ids]

    def encode_batch(self, texts, add_special_tokens=True,
                     num_workers=None, chunk_size=256):
        """Encode many documents, optionally across processes (fork only)."""
        self._prepare_inference()
        texts = list(texts)

        if num_workers is None:
            num_workers = os.cpu_count() or 1

        if num_workers <= 1 or len(texts) < 2 * chunk_size:
            return [self.encode(t, add_special_tokens) for t in texts]

        import multiprocessing as mp
        chunks = [
            (texts[i:i + chunk_size], add_special_tokens)
            for i in range(0, len(texts), chunk_size)
        ]
        ctx = mp.get_context("fork")
        with ctx.Pool(num_workers, initializer=_init_worker,
                      initargs=(self,)) as pool:
            results = list(pool.imap(_encode_chunk, chunks))
        return [ids for chunk in results for ids in chunk]

    def decode(self, token_ids):
        """Decodes a list of token IDs back into a string, handling special tokens."""
        self._prepare_inference()
        # Filter out special tokens for a cleaner decoded string
        filtered_ids = [
            tid for tid in token_ids
            if self.itos[tid] not in self.reserved_tokens
        ]
        # Join the token strings, replacing </w> with a space
        decoded_text = "".join([self.itos[tid] for tid in filtered_ids]).replace(self.end_marker, " ")
        return decoded_text.strip()

In [ ]:
# ============================================================
# normalize_text - copied VERBATIM from the Phase-1 Mamba notebook's cleaning cell.
# Must be identical for training and inference (Phase-5 documents get the same normalisation).
# ============================================================
import re
import zlib
import hashlib
import unicodedata
import collections
import numpy as np


# ------------------------------------------------------------
# 1. TEXT NORMALISATION
#    normalize_text() is also applied to the test prompts at
#    inference time so train and test text look the same.
# ------------------------------------------------------------
# characters that are deleted outright (zero-width, bidi marks, soft hyphen,
# BOM, the U+FFFD "decode error" marker)
_CHAR_MAP = {ord(c): None for c in (
    "\u200b\u200c\u200d\u200e\u200f\u2060\ufeff\u00ad\ufffd"
    "\u202a\u202b\u202c\u202d\u202e"
)}
# typographic quotes / dashes -> ASCII, so one symbol is not split over 3 variants
_CHAR_MAP.update({ord(c): "'" for c in "\u2018\u2019\u201a\u201b\u2032\u02bc"})
_CHAR_MAP.update({ord(c): '"' for c in "\u201c\u201d\u201e\u201f\u2033"})
_CHAR_MAP.update({ord(c): "-" for c in "\u2010\u2011\u2012\u2013\u2014\u2015\u2212"})

_CTRL_RE = re.compile(r"[\x00-\x08\x0e-\x1b\x7f\x80-\x84\x86-\x9f]")
_WS_RE = re.compile(r"\s+")                      # any run of unicode whitespace
_REPEAT_RE = re.compile(r"([^\w\s]|_)\1{3,}")    # '......', '-----', '_____' ...


def normalize_text(text, collapse_repeats=True):
    """Unicode-normalise and collapse whitespace.

    - quotes/dashes -> ASCII, NFKC (turns nbsp, thin/ideographic spaces,
      ligatures, full-width forms into plain characters)
    - removes zero-width / control / replacement characters
    - every run of whitespace (spaces, tabs, newlines, nbsp ...) -> one ' '
    - runs of 4+ identical punctuation chars ('.....', '-----') -> 3
    """
    text = text.translate(_CHAR_MAP)
    if not text.isascii():
        text = unicodedata.normalize("NFKC", text)
    text = _CTRL_RE.sub("", text)
    text = _WS_RE.sub(" ", text).strip()
    if collapse_repeats:
        text = _REPEAT_RE.sub(lambda m: m.group(1) * 3, text)
    return text

In [ ]:
# ============================================================
# 0. SETUP
# ============================================================
import os, json, math, time, random, copy, pickle, shutil, gc, glob, hashlib
from types import SimpleNamespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_ckpt

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "no GPU")
AMP_DTYPE = (None if device.type != "cuda" else
             torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)   # fp16 on T4
print("mixed precision:", AMP_DTYPE)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
else:
    print("WARNING: no HF_TOKEN found - the gated dataset download will fail with a 401.")

# ---- paths (same layout as the Phase-5 Mamba notebook; edit if yours differ) ------------------------
cache_dir = "/content/drive/MyDrive/CS781/Capstone-Project/legal_dataset_cache"
ckpt_path = os.path.join(cache_dir, "mamba_checkpoints", "mamba_large.pt")      # Phase-1 Mamba
tok_path  = os.path.join(cache_dir, "final_bpe_en.json")
p6_dir    = os.path.join(cache_dir, "phase6_mamba"); os.makedirs(p6_dir, exist_ok=True)
local_ck  = "/content/p6_ckpt"; os.makedirs(local_ck, exist_ok=True)

# ---- Mamba hyper-parameters: MUST equal what you pre-trained with -----------------------------------
MAMBA_CFG = dict(d_model=512, n_layers=20, d_state=32, d_conv=16, expand=3, dropout=0.1)
PRETRAIN_LEN = 512                     # Phase-1 training length (Mamba has no hard window, but saw only 512)

tokenizer = BPETokenizer.from_json(tok_path)
PAD_ID, BOS_ID, EOS_ID = tokenizer.stoi["<pad>"], tokenizer.stoi["<s>"], tokenizer.stoi["</s>"]
print(f"tokenizer vocab {len(tokenizer.itos):,} | PAD {PAD_ID} BOS {BOS_ID} EOS {EOS_ID}")

Mounted at /content/drive
device: cuda | NVIDIA L4
mixed precision: torch.bfloat16
tokenizer vocab 32,768 | PAD 0 BOS 2 EOS 3


In [ ]:
# ============================================================
# MAMBA (Gu & Dao, 2023) - the Phase-1 architecture, with an optional fused CUDA scan.
#
# Parameter / module names are IDENTICAL to the Phase-1 notebook's MambaLM, so the Phase-1
# checkpoint loads with strict=True. Differences from Phase 1 (maths unchanged):
#   * forward() can run the fused `mamba_ssm` selective-scan kernel: softplus(dt + bias), the
#     scan, the D*x skip and the SiLU(z) gate are one CUDA launch.
#   * the pure-PyTorch fallback scan now runs in 128-step segments (carrying the SSM state), so
#     memory is bounded by the segment instead of the whole 512-token chunk.
#   * no generate()/step(): Phase 5 only needs hidden states.
# ============================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint

# ---- optional fused kernels ---------------------------------------------------------------
HAS_SSM_KERNEL, HAS_CONV_KERNEL, _KERNEL_ERR = False, False, None
try:
    from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
    HAS_SSM_KERNEL = True
except Exception as _e:                                   # not installed / no matching build
    selective_scan_fn, _KERNEL_ERR = None, repr(_e)
try:
    from causal_conv1d import causal_conv1d_fn
    HAS_CONV_KERNEL = True
except Exception:
    causal_conv1d_fn = None

# runtime switches (the self-test cell may turn the scan kernel off if it disagrees with the reference)
FAST = {
    "scan_kernel": bool(HAS_SSM_KERNEL and torch.cuda.is_available()),
    "conv_kernel": bool(HAS_CONV_KERNEL and torch.cuda.is_available()),
}
print("mamba_ssm selective-scan kernel :", "available" if HAS_SSM_KERNEL else f"NOT available ({_KERNEL_ERR})")
print("causal-conv1d kernel            :", "available" if HAS_CONV_KERNEL else "not available (only usable for d_conv <= 4)")
print("scan kernel in use              :", FAST["scan_kernel"])

_MIN_LOG_DECAY = -8.0            # Phase-1 clamp on per-step log(a_t) = dt * A (pure-PyTorch path only)


# ============================================================
# Pure-PyTorch scan (Phase-1 maths):  h_t = exp(dt_t A) h_{t-1} + dt_t B_t x_t ;  y_t = C_t . h_t + D x_t
# ============================================================
def selective_scan_chunked(x, dt, A, Bm, Cm, D, chunk=8, h0=None):
    """Chunked parallel scan in fp32.  x, dt: (B, L, Din)  A: (Din, N)  Bm, Cm: (B, L, N)  D: (Din,)
    h0: optional initial state (B, Din, N).  Returns y (B, L, Din) and the final state (B, Din, N)."""
    Bsz, L, Din = x.shape
    N = A.shape[1]

    pad = (-L) % chunk                       # zero padding is a no-op for the state (dt = 0 -> a = 1, b = 0)
    if pad:
        x, dt = F.pad(x, (0, 0, 0, pad)), F.pad(dt, (0, 0, 0, pad))
        Bm, Cm = F.pad(Bm, (0, 0, 0, pad)), F.pad(Cm, (0, 0, 0, pad))
    Lp = L + pad
    nc = Lp // chunk

    dA = (dt.unsqueeze(-1) * A).clamp(min=_MIN_LOG_DECAY)        # (B, Lp, Din, N), <= 0
    dBx = (dt * x).unsqueeze(-1) * Bm.unsqueeze(2)               # (B, Lp, Din, N)
    dA = dA.view(Bsz, nc, chunk, Din, N)
    dBx = dBx.view(Bsz, nc, chunk, Din, N)

    La = torch.cumsum(dA, dim=2)                                  # within-chunk cumulative log-decay
    local = torch.exp(La) * torch.cumsum(dBx * torch.exp(-La), dim=2)   # chunk-local states (zero initial state)

    chunk_decay = torch.exp(La[:, :, -1])                         # (B, nc, Din, N)
    chunk_end = local[:, :, -1]                                   # (B, nc, Din, N)

    carry = x.new_zeros(Bsz, Din, N) if h0 is None else h0
    prev = []
    for k in range(nc):                                           # only L/chunk iterations
        prev.append(carry)
        carry = chunk_decay[:, k] * carry + chunk_end[:, k]
    h_last = carry
    prev = torch.stack(prev, dim=1)                               # state entering each chunk

    h = local + torch.exp(La) * prev.unsqueeze(2)                 # (B, nc, chunk, Din, N)
    y = (h * Cm.view(Bsz, nc, chunk, 1, N)).sum(-1).reshape(Bsz, Lp, Din)[:, :L]
    y = y + x[:, :L] * D
    return y, h_last


def selective_scan_segmented(x, dt, A, Bm, Cm, D, chunk=8, seg=128):
    """Same result as selective_scan_chunked, but the (B, seg, Din, N) intermediates only exist for
    `seg` steps at a time (the SSM state is carried between segments). `seg` must be a multiple of `chunk`."""
    L = x.size(1)
    if L <= seg:
        return selective_scan_chunked(x, dt, A, Bm, Cm, D, chunk)
    ys, h = [], None
    for s in range(0, L, seg):
        y, h = selective_scan_chunked(x[:, s:s + seg], dt[:, s:s + seg], A,
                                      Bm[:, s:s + seg], Cm[:, s:s + seg], D, chunk, h0=h)
        ys.append(y)
    return torch.cat(ys, dim=1), h


# ============================================================
# Norm
# ============================================================
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return xf.to(x.dtype) * self.weight


def rms_normalize(x, eps=1e-5):
    """Parameter-free RMS norm (used to read out intermediate blocks, where no learned norm exists)."""
    xf = x.float()
    return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps)).to(x.dtype)


# ============================================================
# Mamba mixer (selective SSM part of the block, Fig. 3 of the paper)
# ============================================================
class MambaMixer(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dt_rank=None,
                 dt_min=1e-3, dt_max=1e-1, dt_init_floor=1e-4, scan_chunk=8):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.d_inner = expand * d_model
        self.dt_rank = math.ceil(d_model / 16) if dt_rank is None else dt_rank
        self.scan_chunk = scan_chunk

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)          # -> (x, z)
        self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, kernel_size=d_conv,
                                groups=self.d_inner, padding=d_conv - 1, bias=True)
        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)  # -> (dt, B, C)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        nn.init.uniform_(self.dt_proj.weight, -self.dt_rank ** -0.5, self.dt_rank ** -0.5)
        dt = torch.exp(torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
                       ).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))                               # softplus^-1
        with torch.no_grad():
            self.dt_proj.bias.copy_(inv_dt)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        self.A_log._no_weight_decay = True
        self.D._no_weight_decay = True

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def _conv_silu(self, x):
        """x: (B, Din, L) -> SiLU(causal depthwise conv(x)), (B, Din, L)."""
        L = x.size(-1)
        if FAST["conv_kernel"] and self.d_conv <= 4 and x.is_cuda:
            return causal_conv1d_fn(x.contiguous(), self.conv1d.weight.squeeze(1), self.conv1d.bias,
                                    activation="silu")
        return F.silu(self.conv1d(x)[..., :L])

    def forward(self, u):
        """u: (B, L, d_model) -> (B, L, d_model)"""
        x, z = self.in_proj(u).chunk(2, dim=-1)                                   # (B, L, Din) each
        x = self._conv_silu(x.transpose(1, 2))                                    # (B, Din, L)
        dt, Bm, Cm = torch.split(self.x_proj(x.transpose(1, 2)),
                                 [self.dt_rank, self.d_state, self.d_state], dim=-1)
        A = -torch.exp(self.A_log.float())                                        # (Din, N)

        if FAST["scan_kernel"] and u.is_cuda:
            # fused kernel: softplus(dt + bias), scan, D*x and the SiLU(z) gate in one launch.
            # (It does not apply Phase-1's -8 clamp on dt*A; that only changes how much of an already
            #  fully-decayed state survives, <= ~3e-4 relative. The self-test cell measures the net effect.)
            dt = F.linear(dt, self.dt_proj.weight)                                # bias is added inside the kernel
            cd = x.dtype                                                          # u, dt, B, C, z must share a dtype
            y = selective_scan_fn(
                x.contiguous(),
                dt.to(cd).transpose(1, 2).contiguous(),
                A,
                Bm.to(cd).transpose(1, 2).contiguous(),
                Cm.to(cd).transpose(1, 2).contiguous(),
                self.D.float(),
                z=z.to(cd).transpose(1, 2).contiguous(),
                delta_bias=self.dt_proj.bias.float(),
                delta_softplus=True,
            )
            y = y.transpose(1, 2)                                                 # (B, L, Din), already gated
        else:
            dt = F.softplus(self.dt_proj(dt))                                     # (B, L, Din)
            with torch.autocast(device_type=u.device.type, enabled=False):        # scan always fp32
                y, _ = selective_scan_segmented(
                    x.transpose(1, 2).float(), dt.float(), A, Bm.float(), Cm.float(),
                    self.D.float(), chunk=self.scan_chunk)
            y = y.to(z.dtype) * F.silu(z)                                         # gate branch
        return self.out_proj(y)


# ============================================================
# Block (pre-norm residual) and the Phase-1 language model (weights are loaded, never re-initialised)
# ============================================================
class MambaBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1, **mixer_kwargs):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.mixer = MambaMixer(d_model, **mixer_kwargs)
        self.dropout = nn.Dropout(dropout)
        self.adapter = None            # optional bottleneck adapter (Phase-5 PEFT); None adds no parameters / state_dict keys

    def forward(self, x):
        y = self.mixer(self.norm(x))
        if self.adapter is not None:
            y = self.adapter(y)
        return x + self.dropout(y)


class MambaLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, d_state=16, d_conv=4, expand=2,
                 dropout=0.1, pad_token_id=0, scan_chunk=8):
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_token_id)
        self.layers = nn.ModuleList([
            MambaBlock(d_model, dropout=dropout, d_state=d_state, d_conv=d_conv,
                       expand=expand, scan_chunk=scan_chunk)
            for _ in range(n_layers)
        ])
        self.norm_final = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)   # only so the checkpoint loads strictly

    def forward(self, input_ids):                                    # full LM logits (not used in Phase 5)
        x = self.token_embedding(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.norm_final(x))

mamba_ssm selective-scan kernel : NOT available (ModuleNotFoundError("No module named 'mamba_ssm'"))
causal-conv1d kernel            : not available (only usable for d_conv <= 4)
scan kernel in use              : False


In [ ]:
# ============================================================
# 1. MAMBA FOR GENERATION (same weights and maths as Phase 1; replaces MambaMixer / MambaBlock forward)
#
#   * key-padding for LEFT-padded prefixes: the in_proj output and the conv output are zeroed at padding positions.
#     The SSM state then stays exactly 0 through the padding (zero input, decay of a zero state) and the causal
#     conv sees zeros, which is what it sees at the start of a sequence anyway. So a left-padded row behaves exactly
#     like the unpadded one. (Right padding after the target is harmless: the model is causal.)
#   * return_state: after a prefix, return (conv window, SSM state) per block, for recurrent decoding.
#   * mixer_step: one-token recurrent update, O(1) per token (no KV cache growth, no context limit).
# ============================================================
MAMBA_CTX = {"mask": None}            # [B, L] bool, True = real position; set by mamba_seq()

if not hasattr(MambaMixer, "_orig_forward"):
    MambaMixer._orig_forward = MambaMixer.forward
    MambaBlock._orig_forward = MambaBlock.forward


def mixer_forward(self, u, return_state=False):
    mask = MAMBA_CTX["mask"]
    L = u.size(1)
    xz = self.in_proj(u)
    if mask is not None:
        xz = xz * mask[:, :, None].to(xz.dtype)
    x, z = xz.chunk(2, dim=-1)                                         # (B, L, Din)
    xc_in = x.transpose(1, 2)                                          # conv input (B, Din, L)
    x = self._conv_silu(xc_in)
    if mask is not None:
        x = x * mask[:, None, :].to(x.dtype)
    dt, Bm, Cm = torch.split(self.x_proj(x.transpose(1, 2)), [self.dt_rank, self.d_state, self.d_state], dim=-1)
    A = -torch.exp(self.A_log.float())
    h_last = None
    if FAST["scan_kernel"] and u.is_cuda:
        dt = F.linear(dt, self.dt_proj.weight)
        cd = x.dtype
        res = selective_scan_fn(x.contiguous(), dt.to(cd).transpose(1, 2).contiguous(), A,
                                Bm.to(cd).transpose(1, 2).contiguous(), Cm.to(cd).transpose(1, 2).contiguous(),
                                self.D.float(), z=z.to(cd).transpose(1, 2).contiguous(),
                                delta_bias=self.dt_proj.bias.float(), delta_softplus=True,
                                return_last_state=return_state)
        y, h_last = res if return_state else (res, None)
        y = y.transpose(1, 2)
    else:
        dt = F.softplus(self.dt_proj(dt))
        with torch.autocast(device_type=u.device.type, enabled=False):
            y, h_last = selective_scan_segmented(x.transpose(1, 2).float(), dt.float(), A, Bm.float(), Cm.float(),
                                                 self.D.float(), chunk=self.scan_chunk)
        y = y.to(z.dtype) * F.silu(z)
    out = self.out_proj(y)
    if not return_state:
        return out
    k = self.d_conv - 1
    conv = xc_in[..., -k:] if L >= k else F.pad(xc_in, (k - L, 0))
    return out, (conv, h_last.float())


def mixer_step(self, u, state):
    """u [B, d_model] (one token), state = (conv window [B, Din, d_conv-1], SSM state [B, Din, N] fp32)"""
    conv, h = state
    x, z = self.in_proj(u).chunk(2, dim=-1)                           # [B, Din]
    win = torch.cat([conv.to(x.dtype), x[:, :, None]], -1)             # [B, Din, d_conv]
    xc = (win * self.conv1d.weight[:, 0, :].to(x.dtype)).sum(-1) + self.conv1d.bias.to(x.dtype)
    x = F.silu(xc)
    dt, Bm, Cm = torch.split(self.x_proj(x), [self.dt_rank, self.d_state, self.d_state], dim=-1)
    with torch.autocast(device_type=u.device.type, enabled=False):
        A = -torch.exp(self.A_log.float())
        dt = F.softplus(F.linear(dt.float(), self.dt_proj.weight.float(), self.dt_proj.bias.float()))   # [B, Din]
        dA = dt[:, :, None] * A
        if not (FAST["scan_kernel"] and u.is_cuda):                    # Phase-1 clamp (pure path only, as in forward)
            dA = dA.clamp(min=_MIN_LOG_DECAY)
        xf = x.float()
        h = torch.exp(dA) * h + (dt * xf)[:, :, None] * Bm.float()[:, None, :]
        y = (h * Cm.float()[:, None, :]).sum(-1) + self.D.float() * xf
    y = y.to(z.dtype) * F.silu(z)
    return self.out_proj(y), (win[..., 1:], h)


def block_forward(self, x, return_state=False):
    r = self.mixer(self.norm(x), return_state=return_state)
    y, st = r if return_state else (r, None)
    if self.adapter is not None:
        y = self.adapter(y)
    x = x + self.dropout(y)
    return (x, st) if return_state else x


def block_step(self, x, state):
    y, st = mixer_step(self.mixer, self.norm(x), state)
    if self.adapter is not None:
        y = self.adapter(y)
    return x + self.dropout(y), st


MambaMixer.forward = mixer_forward
MambaBlock.forward = block_forward


def mamba_seq(model, ids=None, embeds=None, mask=None, return_state=False, ckpt=False, upto=None):
    """Full-sequence pass -> final hidden states [B,L,d] (after norm_final), and per-block states if asked.
    embeds: soft inputs instead of ids; mask: [B,L] bool for LEFT padding; upto: stop after block `upto`
    (parameter-free RMS read-out). Bottom blocks with nothing trainable run under no_grad."""
    h = model.token_embedding(ids) if embeds is None else embeds
    layers = model.layers if upto is None else model.layers[:upto + 1]
    lo = 0
    if torch.is_grad_enabled() and not h.requires_grad:
        while lo < len(layers) and not any(p.requires_grad for p in layers[lo].parameters()):
            lo += 1
    states = []
    for i, blk in enumerate(layers):
        def run(x, blk=blk, m=mask):
            MAMBA_CTX["mask"] = m
            try:
                return blk(x, return_state=return_state)
            finally:
                MAMBA_CTX["mask"] = None
        if i < lo:
            with torch.no_grad():
                r = run(h)
        elif ckpt and torch.is_grad_enabled() and not return_state:
            r = grad_ckpt(run, h, use_reentrant=False)
        else:
            r = run(h)
        if return_state:
            h, st = r; states.append(st)
        else:
            h = r
    h = model.norm_final(h) if upto is None else rms_normalize(h)
    return h, (states if return_state else None)


def mamba_step(model, tok, states):
    """one token for every row: tok [B] -> (final hidden [B,d], new states)"""
    h = model.token_embedding(tok)
    new = []
    for blk, st in zip(model.layers, states):
        h, s = block_step(blk, h, st)
        new.append(s)
    return model.norm_final(h), new


def reorder_states(states, idx):
    return [(c.index_select(0, idx), s.index_select(0, idx)) for c, s in states]


def load_pretrained_mamba():
    m = MambaLM(vocab_size=len(tokenizer.itos), pad_token_id=PAD_ID, **MAMBA_CFG)
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    m.load_state_dict(ck["model_state_dict"])          # strict; the LM head is KEPT (it writes the summary)
    return m.eval()


MAMBA_BASE = load_pretrained_mamba()
print(f"Mamba loaded: {sum(p.numel() for p in MAMBA_BASE.parameters())/1e6:.1f}M params | "
      f"{len(MAMBA_BASE.layers)} blocks | scan kernel in use: {FAST['scan_kernel']}")

Mamba loaded: 86.2M params | 20 blocks | scan kernel in use: False


In [ ]:
# ============================================================
# DATA: Phase-6 (abstractive summarisation) from the course Hugging Face repo
# ============================================================
from huggingface_hub import hf_hub_download, HfApi

REPO_ID = "Exploration-Lab/CS-781-Capstone"


def find_repo_file(basename):
    """Locate e.g. train.json inside the Phase-6 folder without hard-coding its exact name."""
    files = HfApi().list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)
    hits = [f for f in files if f.endswith("/" + basename) and ("summ" in f.lower() or "phase_6" in f.lower())]
    if len(hits) != 1:
        raise RuntimeError(f"expected exactly one {basename} for Phase 6, got {hits}. "
                           f"Set the path by hand. Phase-6-like files: {[f for f in files if 'summ' in f.lower()]}")
    return hits[0]


_DOC_KEYS = ("document", "text", "judgment", "judgement", "doc", "source", "input", "article", "case_text")
_SUM_KEYS = ("summary", "summaries", "abstract", "target", "reference", "headnote", "output", "gold", "label")


def _as_text(v):
    if v is None:
        return None
    if isinstance(v, list):
        v = " ".join(str(x) for x in v)
    v = str(v).strip()
    return v or None


def read_split(filename):
    path = hf_hub_download(REPO_ID, find_repo_file(filename), repo_type="dataset", token=HF_TOKEN)
    raw = json.load(open(path, encoding="utf-8"))
    if isinstance(raw, dict):
        raw = [({**v, "id": k} if isinstance(v, dict) else {"id": k, "document": v}) for k, v in raw.items()]
    keys = list(raw[0].keys())
    print(f"{filename}: {len(raw)} records | fields: {keys}")
    # document = first known doc key present, else the longest text field; summary = first known summary key present
    doc_k = next((k for k in _DOC_KEYS if k in keys), None)
    sum_k = next((k for k in _SUM_KEYS if k in keys), None)
    if doc_k is None:
        text_keys = [k for k in keys if k != "id" and isinstance(raw[0][k], (str, list))]
        doc_k = max(text_keys, key=lambda k: len(_as_text(raw[0][k]) or ""))
    print(f"  using document field '{doc_k}', summary field '{sum_k}'")
    return [dict(id=str(r.get("id", i)), doc=_as_text(r.get(doc_k)) or "",
                 summ=_as_text(r.get(sum_k)) if sum_k else None) for i, r in enumerate(raw)]


all_train_recs = read_split("train.json")
test_recs = read_split("test.json")
_n_missing = sum(r["summ"] is None for r in all_train_recs)
if _n_missing == len(all_train_recs):
    raise KeyError("no summary field found in train.json: look at the 'fields' printed above and add the "
                   "summary field's name to _SUM_KEYS")
if _n_missing:
    print(f"dropping {_n_missing} training records with an empty summary")
all_train_recs = [r for r in all_train_recs if r["summ"] is not None and r["doc"]]
print(f"{len(all_train_recs)} train documents | {len(test_recs)} test documents")

# ---- no official dev split for Phase 6: hold out a validation split from train ----------------
VAL_FRAC, SPLIT_SEED = 0.05, 1234
_perm = np.random.RandomState(SPLIT_SEED).permutation(len(all_train_recs))
_n_val = max(50, int(round(VAL_FRAC * len(all_train_recs))))
val_recs = [all_train_recs[i] for i in _perm[:_n_val]]
train_recs = [all_train_recs[i] for i in _perm[_n_val:]]
print(f"split -> train {len(train_recs)} | val {len(val_recs)}  (val is held out from train.json)")

# ---- tokenise once, cache to Drive (no special tokens: added per chunk / per sequence later) ----
# NORMALIZE_TARGETS: apply the Phase-1 normalize_text to the reference summaries too. Keep True if
# normalize_text only cleans whitespace/unicode; set False if it lower-cases (the scorer compares
# against the raw references, so a lower-cased output would cost BERTScore).
NORMALIZE_TARGETS = True


def _tok_cached(texts, tag, norm=True, bs=1000, workers=None):
    workers = workers or min(4, os.cpu_count() or 1)
    path = os.path.join(p6_dir, f"tok_{tag}.pkl")
    if os.path.exists(path):
        return pickle.load(open(path, "rb"))
    out = []
    for i in range(0, len(texts), bs):
        part = [normalize_text(t) if norm else t for t in texts[i:i + bs]]
        out += [np.asarray(x, dtype=np.int32) for x in
                tokenizer.encode_batch(part, add_special_tokens=False, num_workers=workers, chunk_size=16)]
        print(f"  tokenised {min(i + bs, len(texts)):,}/{len(texts):,} ({tag})")
    pickle.dump(out, open(path, "wb"))
    return out


def _tok_split(recs, tag):
    docs = _tok_cached([r["doc"] for r in recs], f"{tag}_doc_s{SPLIT_SEED}")
    sums = None
    if recs[0]["summ"] is not None:
        sums = _tok_cached([r["summ"] for r in recs], f"{tag}_sum_s{SPLIT_SEED}_n{int(NORMALIZE_TARGETS)}",
                           norm=NORMALIZE_TARGETS)
    return docs, sums


tr_doc, tr_sum = _tok_split(train_recs, "train")
va_doc, va_sum = _tok_split(val_recs, "val")
te_doc, _ = _tok_split(test_recs, "test")

# ---- length statistics: these decide the generation lengths and the context layout ------------
dl = np.array([len(x) for x in tr_doc]); sl = np.array([len(x) for x in tr_sum])
sw = np.array([len(r["summ"].split()) for r in train_recs])
pct = lambda a: "  ".join(f"p{p} {int(np.percentile(a, p))}" for p in (10, 50, 90, 95, 99))
print("document BPE tokens :", pct(dl), f" max {dl.max()}")
print("summary  BPE tokens :", pct(sl), f" max {sl.max()}")
print("summary  words      :", pct(sw))
print(f"tokens per summary word ~ {sl.sum() / max(sw.sum(), 1):.2f} | summary/document length ratio median "
      f"{np.median(sl / np.maximum(dl, 1)):.3f}")
print(f"512-token chunks per document: median {np.median(np.ceil(dl / 511)):.0f}  p90 {np.percentile(np.ceil(dl / 511), 90):.0f}")

SUM_TOK_P10, SUM_TOK_P50, SUM_TOK_P90, SUM_TOK_P95 = (int(np.percentile(sl, p)) for p in (10, 50, 90, 95))
SUM_WORDS_P50 = int(np.median(sw))

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

train.json: 7030 records | fields: ['id', 'document', 'summary']
  using document field 'document', summary field 'summary'


test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

test.json: 100 records | fields: ['id', 'document']
  using document field 'document', summary field 'None'
dropping 2 training records with an empty summary
7028 train documents | 100 test documents
split -> train 6677 | val 351  (val is held out from train.json)
document BPE tokens : p10 1575  p50 3667  p90 9124  p95 13101  p99 28218  max 162967
summary  BPE tokens : p10 342  p50 753  p90 1774  p95 2399  p99 5114  max 32546
summary  words      : p10 292  p50 638  p90 1493  p95 2007  p99 4267
tokens per summary word ~ 1.19 | summary/document length ratio median 0.220
512-token chunks per document: median 8  p90 18


In [ ]:
# ============================================================
# Model-agnostic utilities: chunking, detokenisation, post-processing, ROUGE / BERTScore, submission
# ============================================================
import re, zipfile, collections


# ---------------- long-document chunking (as in Phase 5) ----------------
def chunk_ids(ids, body_len=511, stride=511, min_tail=16):
    if len(ids) <= body_len:
        return [ids]
    chunks, s = [], 0
    while True:
        chunks.append(ids[s:s + body_len])
        if s + body_len >= len(ids):
            break
        s += stride
    if len(chunks) > 1 and len(chunks[-1]) < min_tail:
        chunks.pop()
    return chunks


def select_chunks(chunks, max_chunks, strategy="uniform"):
    """uniform: spread the budget over the whole judgment (facts at the start, reasoning in the middle,
    the order at the end all matter for a summary). head / tail / head_tail kept for the ablation."""
    n = len(chunks)
    if n <= max_chunks:
        return chunks
    if strategy == "head":
        return chunks[:max_chunks]
    if strategy == "tail":
        return chunks[-max_chunks:]
    if strategy == "head_tail":
        t = max_chunks // 2
        return chunks[:max_chunks - t] + (chunks[-t:] if t else [])
    if strategy == "uniform":
        idx = np.unique(np.linspace(0, n - 1, max_chunks).round().astype(int))
        return [chunks[i] for i in idx]
    raise ValueError(strategy)


# ---------------- detokenisation + post-processing ----------------
_FIXES = [(re.compile(r"\s+([.,;:!?%)\]}'])"), r"\1"),      # 'court .'  -> 'court.'
          (re.compile(r"([(\[{])\s+"), r"\1"),               # '( a )'    -> '(a)'
          (re.compile(r"(?<=\w)\s*([-/])\s*(?=\w)"), r"\1"),  # '302 / 34' -> '302/34', 'Section - 3' -> 'Section-3'
          (re.compile(r"\s{2,}"), " ")]


def detok(ids):
    """BPE ids -> text. tokenizer.decode() puts a space after every word-final token including punctuation
    ('court . the'), so spacing is repaired here. Special tokens are dropped."""
    ids = [int(i) for i in ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
    s = tokenizer.decode(ids)
    for pat, rep in _FIXES:
        s = pat.sub(rep, s)
    return s.strip()


def postprocess(text, max_words=None):
    """drop exact/near-exact repeated sentences (small models loop), cap length, end on a sentence boundary"""
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    seen, out = set(), []
    for s in sents:
        key = re.sub(r"\W+", " ", s.lower()).strip()
        if len(key) < 2 or key in seen:
            continue
        seen.add(key); out.append(s)
    text = " ".join(out)
    if max_words:
        w = text.split()
        if len(w) > max_words:
            text = " ".join(w[:max_words])
    cut = max(text.rfind(". "), text.rfind("."))
    if cut > 0.8 * len(text) and cut < len(text) - 1:     # trailing fragment after the last full stop
        text = text[:cut + 1]
    return text.strip() or "."


# ---------------- metrics ----------------
try:
    from rouge_score import rouge_scorer as _rs
    _ROUGE = _rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
except Exception:
    _ROUGE = None


def _toks(s):
    return re.findall(r"[a-z0-9]+", s.lower())


def _f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def _rouge_n(a, b, n):
    ca = collections.Counter(zip(*[a[i:] for i in range(n)])); cb = collections.Counter(zip(*[b[i:] for i in range(n)]))
    ov = sum((ca & cb).values())
    return _f1(ov / max(sum(ca.values()), 1), ov / max(sum(cb.values()), 1))


def _rouge_l(a, b):
    if not a or not b:
        return 0.0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    l = prev[-1]
    return _f1(l / len(a), l / len(b))


def rouge_scores(preds, refs):
    r1 = r2 = rl = 0.0
    for p, r in zip(preds, refs):
        if _ROUGE is not None:
            s = _ROUGE.score(r, p)
            r1 += s["rouge1"].fmeasure; r2 += s["rouge2"].fmeasure; rl += s["rougeL"].fmeasure
        else:                                            # fallback (no stemming): numbers differ slightly
            a, b = _toks(p), _toks(r)
            r1 += _rouge_n(a, b, 1); r2 += _rouge_n(a, b, 2); rl += _rouge_l(a, b)
    n = max(len(preds), 1)
    return dict(rouge1=r1 / n, rouge2=r2 / n, rougeL=rl / n)


BERTSCORE_KW = dict(lang="en")      # the course scorer's settings are not published; lang="en" = roberta-large
_BS = {}


def bertscore_f1(preds, refs, batch_size=16):
    """Evaluation only (the official scorer does the same). Returns None if bert-score is unavailable."""
    try:
        from bert_score import BERTScorer
    except Exception:
        return None
    key = json.dumps(BERTSCORE_KW, sort_keys=True)
    if key not in _BS:
        _BS[key] = BERTScorer(device=str(device), **BERTSCORE_KW)
    _, _, f = _BS[key].score(preds, refs, batch_size=batch_size)
    return float(f.mean())


def evaluate_texts(preds, refs, bert=True):
    m = rouge_scores(preds, refs)
    m["bertscore"] = bertscore_f1(preds, refs) if bert else None
    m["avg_words"] = float(np.mean([len(p.split()) for p in preds]))
    m["ref_words"] = float(np.mean([len(r.split()) for r in refs]))
    return m


def fmt_metrics(m):
    bs = "n/a" if m.get("bertscore") is None else f"{m['bertscore']:.4f}"
    return (f"BERTScore {bs} | R1 {m['rouge1']:.4f} R2 {m['rouge2']:.4f} RL {m['rougeL']:.4f} | "
            f"words {m['avg_words']:.0f} (ref {m['ref_words']:.0f})")


# ---------------- submission ----------------
def write_submission(preds_by_id, name):
    out = [dict(id=r["id"], summary=preds_by_id[r["id"]]) for r in test_recs]
    assert len(out) == len(test_recs) == len({o["id"] for o in out}), "need exactly one summary per test id"
    assert all(isinstance(o["summary"], str) and o["summary"].strip() for o in out), "empty summary"
    d = os.path.join(p6_dir, f"submission_{name}"); os.makedirs(d, exist_ok=True)
    jp = os.path.join(d, "prediction.json")
    json.dump(out, open(jp, "w", encoding="utf-8"), ensure_ascii=False)
    zp = os.path.join(d, "submission.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")          # at the ROOT of the zip, as Codabench requires
    print(f"wrote {len(out)} summaries (avg {np.mean([len(o['summary'].split()) for o in out]):.0f} words) -> {zp}")
    return zp

In [ ]:
# ============================================================
# Generic batched beam search (model-agnostic)
#
# The model side supplies three closures:
#   start()            -> (state, logits[B,V])   consume the prompt / encode the source, logits for the 1st token
#   step(state, tok)   -> (state, logits[N,V])   consume one token per hypothesis (N = B*beams)
#   reorder(state, ix) -> state                  keep hypothesis rows ix (rows never move to another source:
#                                                row r always belongs to source r // beams)
# Features: length penalty, min length, no-repeat-ngram blocking, repetition penalty, early stopping.
# num_beams=1 gives greedy decoding.
# ============================================================
@torch.no_grad()
def beam_search(start, step, reorder, num_beams=4, max_new=256, min_new=0, len_penalty=1.0,
                no_repeat_ngram=3, rep_penalty=1.0, eos_id=None, banned=()):
    eos_id = EOS_ID if eos_id is None else eos_id
    state, logits = start()
    B, V = logits.shape
    K, dev = num_beams, logits.device
    state = reorder(state, torch.arange(B, device=dev).repeat_interleave(K))
    logits = logits.repeat_interleave(K, 0)
    beam_scores = torch.zeros(B, K, device=dev)
    beam_scores[:, 1:] = -1e9                       # identical beams at t=0: keep only the first
    seqs = [[] for _ in range(B * K)]
    finished = [[] for _ in range(B)]               # (normalised score, tokens)
    done = [False] * B
    banned = [t for t in banned if t is not None and t != eos_id]

    for t in range(max_new):
        lp = torch.log_softmax(logits.float(), -1)
        if banned:
            lp[:, banned] = -float("inf")
        if t < min_new:
            lp[:, eos_id] = -float("inf")
        if rep_penalty != 1.0 and t > 0:
            for r, s in enumerate(seqs):
                if s:
                    ix = torch.tensor(sorted(set(s)), device=dev)
                    lp[r, ix] -= math.log(rep_penalty)
        n = no_repeat_ngram
        if n and t >= n - 1:
            for r, s in enumerate(seqs):
                if done[r // K]:
                    continue
                prefix = tuple(s[len(s) - n + 1:]) if n > 1 else ()
                bad = [s[i + n - 1] for i in range(len(s) - n + 1) if tuple(s[i:i + n - 1]) == prefix]
                if bad:
                    lp[r, bad] = -float("inf")

        cand = (beam_scores.view(-1, 1) + lp).view(B, K * V)
        top_s, top_i = cand.topk(2 * K, dim=1)
        top_s, top_i = top_s.tolist(), top_i.tolist()
        sel_rows, sel_tok, new_scores = [], [], []
        for b in range(B):
            chosen = []
            if not done[b]:
                for sc, ix in zip(top_s[b], top_i[b]):
                    beam, tok = divmod(ix, V)
                    if sc < -1e8:
                        break
                    if tok == eos_id:
                        finished[b].append((sc / ((t + 1) ** len_penalty), list(seqs[b * K + beam])))
                    else:
                        chosen.append((b * K + beam, tok, sc))
                    if len(chosen) == K:
                        break
                # early stop: K finished hypotheses and no live beam can beat the worst of them
                if len(finished[b]) >= K:
                    best_live = max((c[2] for c in chosen), default=-1e9) / ((t + 1) ** len_penalty)
                    if best_live <= sorted(f[0] for f in finished[b])[-K]:
                        done[b] = True
            while len(chosen) < K:                    # finished source or not enough candidates: dummy rows
                chosen.append((b * K, eos_id if done[b] else (chosen[0][1] if chosen else eos_id), -1e9))
            for row, tok, sc in chosen:
                sel_rows.append(row); sel_tok.append(tok); new_scores.append(sc)
        if all(done):
            break
        seqs = [seqs[r] + [tk] for r, tk in zip(sel_rows, sel_tok)]
        beam_scores = torch.tensor(new_scores, device=dev).view(B, K)
        rows = torch.tensor(sel_rows, device=dev)
        state = reorder(state, rows)
        state, logits = step(state, torch.tensor(sel_tok, device=dev))

    out = []
    L = len(seqs[0]) if seqs else 0
    for b in range(B):
        if not done[b]:
            for k in range(K):
                sc = float(beam_scores[b, k])
                if sc > -1e8:
                    finished[b].append((sc / (max(L, 1) ** len_penalty), seqs[b * K + k]))
        out.append(max(finished[b], key=lambda x: x[0])[1] if finished[b] else [])
    return out

In [ ]:
# ============================================================
# Parameter-efficient adaptation modules (own implementations, no peft library)
#   LoRA     (Hu et al., 2021)            W x + (a/r) B A x                      B = 0 at init
#   LoRMA+   (Bihany, Patel, Modi, 2025)   (I + (a/r) B A) W0 x                   multiplicative, additive rank
#                                          inflation; B = 0 at init -> identity; computed as y0 + s*B(A y0),
#                                          y0 = W0 x, so no d x d matrix is ever formed (the paper's re-ordering)
#   QLoRA    (Dettmers et al., 2023)       frozen base stored as 4-bit NF4 (blockwise absmax), LoRA on top
#   Adapters (Houlsby et al., 2019 / Pfeiffer et al., 2021)  bottleneck MLP, zero-init up-projection
#   Prefix   (Li & Liang, 2021)            learned per-layer key/value prefixes (built in the model code)
#   Prompt   (Lester et al., 2021)         learned soft input tokens (built in the model code)
# ============================================================
class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype))


class LoRMALinear(nn.Module):
    """LoRMA+ : W = (I_out + (a/r) B A) W0 ;  B in R^{out x r}, A in R^{r x out}. The bias is added after the
    multiplicative transform (it is not part of W0)."""

    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, self.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(self.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        y0 = F.linear(x, self.base.weight.to(x.dtype))
        y = y0 + self.scale * F.linear(F.linear(self.drop(y0), self.A.to(y0.dtype)), self.B.to(y0.dtype))
        return y if self.base.bias is None else y + self.base.bias.to(y.dtype)


NF4_CODE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
                         -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
                         0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
                         0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored as 4-bit NormalFloat codes (2 per byte) + one fp16 absmax per block of 64.
    Dequantised on the fly in forward (memory saving of QLoRA, without bitsandbytes)."""

    def __init__(self, base, block=64):
        super().__init__()
        W = base.weight.detach().float()
        self.shape, self.block = tuple(W.shape), block
        flat = W.reshape(-1)
        pad = (-flat.numel()) % block
        flat = torch.cat([flat, flat.new_zeros(pad)]).view(-1, block)
        absmax = flat.abs().amax(1, keepdim=True).clamp_min(1e-8)
        code = NF4_CODE.to(flat.device)
        idx = torch.empty(flat.shape, dtype=torch.uint8, device=flat.device)
        for i in range(0, flat.size(0), 4096):                       # chunked argmin keeps memory small
            idx[i:i + 4096] = ((flat[i:i + 4096] / absmax[i:i + 4096]).unsqueeze(-1) - code).abs().argmin(-1)
        idx = idx.view(-1)
        if idx.numel() % 2:
            idx = torch.cat([idx, idx.new_zeros(1)])
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.half())
        self.register_buffer("code", code.clone())
        self.n = W.numel()
        self.bias = None if base.bias is None else nn.Parameter(base.bias.detach().clone(), requires_grad=False)
        self.in_features, self.out_features = self.shape[1], self.shape[0]

    def dequant(self, dtype):
        idx = torch.stack([self.packed >> 4, self.packed & 15], 1).view(-1).long()
        w = self.code[idx][: self.absmax.numel() * self.block].view(-1, self.block) * self.absmax.float()
        return w.view(-1)[: self.n].view(self.shape).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), None if self.bias is None else self.bias.to(x.dtype))


class Adapter(nn.Module):
    """bottleneck adapter: x + up(act(down(x))); up is zero-initialised so the model starts unchanged"""

    def __init__(self, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.down, self.up = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d)
        self.drop = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.drop(self.up(F.gelu(self.down(x))))


class AdapterAfter(nn.Module):
    """Applies an adapter to the output of a sub-layer (Houlsby: after attention and FFN; Pfeiffer: after FFN).
    Works for sub-layers that return a tensor or a tuple (tensor, cache)."""

    def __init__(self, inner, d, bottleneck=64, dropout=0.1):
        super().__init__()
        self.inner, self.adapter = inner, Adapter(d, bottleneck, dropout)

    def forward(self, *a, **k):
        out = self.inner(*a, **k)
        if isinstance(out, tuple):
            return (self.adapter(out[0]),) + tuple(out[1:])
        return self.adapter(out)


def replace_linears(root, make, keep):
    """Replace every nn.Linear child c (full name n) under root with make(c) when keep(n) is True."""
    done = []
    for name, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            full = f"{name}.{cname}" if name else cname
            if type(child) is nn.Linear and keep(full):
                setattr(mod, cname, make(child)); done.append(full)
    return done


def n_params(m, trainable=True):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False

In [ ]:
# ============================================================
# Training / evaluation loop (shared by every method; the model file supplies the Summarizer)
#
# A Summarizer exposes:
#   sm.loss(batch, teacher=None)      -> scalar loss (teacher != None -> knowledge distillation)
#   sm.generate(batch, gen)           -> list of generated token-id lists
#   sm.pretrained_param_ids           -> ids of parameters that came from the Phase-1 checkpoint
# and the notebook supplies make_loader(split, cfg, shuffle, gen=False).
# ============================================================
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def amp_ctx(on=True):
    return torch.autocast(device.type, dtype=AMP_DTYPE or torch.bfloat16, enabled=bool(on and AMP_DTYPE is not None))


def trainable_state(sm):
    return {n: p.detach().cpu().clone() for n, p in sm.named_parameters() if p.requires_grad}


def save_trainable(sm, path, extra=None):
    torch.save({"trainable": trainable_state(sm), **(extra or {})}, path)


def load_trainable(sm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = [n for n, p in sm.named_parameters() if p.requires_grad and n not in ck["trainable"]]
    if missing:
        raise RuntimeError(f"checkpoint {path} lacks {len(missing)} trainable tensors, e.g. {missing[:3]} "
                           f"(was it saved with the same method/config?)")
    sm.load_state_dict(ck["trainable"], strict=False)
    return ck


@torch.no_grad()
def eval_loss(sm, loader, max_batches=None):
    sm.eval(); tot, n = 0.0, 0
    for i, b in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        with amp_ctx():
            tot += float(sm.loss(b)); n += 1
    return tot / max(n, 1)


@torch.no_grad()
def generate_split(sm, split, cfg, n_docs=None, gen=None):
    """-> {id: summary text}"""
    sm.eval()
    gen = {**GEN, **(gen or {})}
    loader = make_loader(split, cfg, shuffle=False, gen=True, limit=n_docs)
    out, t0 = {}, time.time()
    for b in loader:
        with amp_ctx():
            seqs = sm.generate(b, gen)
        for rid, s in zip(b["ids"], seqs):
            out[rid] = postprocess(detok(s), gen.get("max_words"))
        if len(out) % (len(b["ids"]) * 10) == 0:
            print(f"    generated {len(out)} ({time.time() - t0:.0f}s)")
    return out


def score_split(preds, recs, bert=True):
    refs = {r["id"]: r["summ"] for r in recs}
    ids = [i for i in preds if i in refs]
    return evaluate_texts([preds[i] for i in ids], [refs[i] for i in ids], bert=bert)


def train_summarizer(sm, cfg, tag, teacher=None):
    tr, va = make_loader("train", cfg, shuffle=True), make_loader("val", cfg, shuffle=False)
    pre = sm.pretrained_param_ids
    new_p = [p for p in sm.parameters() if p.requires_grad and id(p) not in pre]
    old_p = [p for p in sm.parameters() if p.requires_grad and id(p) in pre]
    groups = ([dict(params=new_p, lr=cfg.lr)] if new_p else []) + ([dict(params=old_p, lr=cfg.lr_backbone)] if old_p else [])
    if not groups:
        raise RuntimeError("nothing to train")
    opt = torch.optim.AdamW(groups, weight_decay=cfg.wd, betas=(0.9, 0.98))
    total = max(1, cfg.epochs * math.ceil(len(tr) / cfg.accum))
    warm = max(1, int(cfg.warmup * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))))
    use_scaler = AMP_DTYPE == torch.float16
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    best, bad, step = -1.0, 0, 0
    ck_path = os.path.join(local_ck, f"{tag}.pt")
    if teacher is not None:
        teacher.eval(); freeze(teacher)
    print(f"[{tag}] {len(tr)} batches/epoch x {cfg.epochs} epochs | accum {cfg.accum} | "
          f"trainable new {sum(p.numel() for p in new_p)/1e6:.2f}M + backbone {sum(p.numel() for p in old_p)/1e6:.2f}M")
    for ep in range(cfg.epochs):
        sm.train(); opt.zero_grad(set_to_none=True)
        t0, run = time.time(), 0.0
        for i, b in enumerate(tr):
            with amp_ctx():
                loss = sm.loss(b, teacher=teacher)
            if not torch.isfinite(loss):
                print(f"[{tag}] non-finite loss at batch {i}, skipped"); opt.zero_grad(set_to_none=True); continue
            scaler.scale(loss / cfg.accum).backward()
            run += float(loss)
            if (i + 1) % cfg.accum == 0 or i + 1 == len(tr):
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg.clip)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if (i + 1) % cfg.log_every == 0:
                print(f"[{tag}] ep {ep+1} batch {i+1}/{len(tr)} loss {run/(i+1):.4f} ({time.time()-t0:.0f}s)")
        vl = eval_loss(sm, va, max_batches=cfg.val_loss_batches)
        preds = generate_split(sm, "val", cfg, n_docs=cfg.n_eval_gen)
        m = score_split(preds, val_recs, bert=cfg.select_metric == "bertscore")
        score = m[cfg.select_metric] if m.get(cfg.select_metric) is not None else m["rougeL"]
        print(f"[{tag}] ep {ep+1:02d} train {run/len(tr):.4f} | val loss {vl:.4f} | {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
        if score > best:
            best, bad = score, 0
            save_trainable(sm, ck_path, {"cfg": vars(cfg), "epoch": ep + 1, "val": m})
        else:
            bad += 1
            if bad >= cfg.patience:
                print(f"[{tag}] early stop"); break
    load_trainable(sm, ck_path)
    shutil.copy(ck_path, os.path.join(p6_dir, f"ckpt_{tag}.pt"))          # persist the best checkpoint to Drive
    return best

In [ ]:
# ============================================================
# 1b. SELF-TEST (seconds). Proves: the patched forward == the Phase-1 forward; a left-padded row == the unpadded
#     row; prefix state + recurrent steps == one full forward; and (if installed) kernel vs pure-PyTorch scan.
# ============================================================
@torch.no_grad()
def _orig_hidden(m, ids):
    MambaMixer.forward, MambaBlock.forward = MambaMixer._orig_forward, MambaBlock._orig_forward
    try:
        h = m.token_embedding(ids)
        for blk in m.layers:
            h = blk(h)
        return m.norm_final(h)
    finally:
        MambaMixer.forward, MambaBlock.forward = mixer_forward, block_forward


def _tol(ref):
    return 2e-3 * max(1.0, ref.abs().max().item())


@torch.no_grad()
def check_mamba_patch():
    m = MAMBA_BASE.to(device).eval()
    docs = [d for d in tr_doc if len(d) >= 300][:4]
    ids = torch.as_tensor(np.stack([d[:200] for d in docs]), dtype=torch.long, device=device)
    B = ids.size(0)

    ref = _orig_hidden(m, ids)
    new, _ = mamba_seq(m, ids)
    d1 = (ref - new).abs().max().item()
    print(f"(1) patched vs Phase-1 forward          max|diff| {d1:.2e}")
    assert d1 < _tol(ref), "patched Mamba forward differs from the Phase-1 forward"

    Tp, Tg, pads = 96, 24, [0, 5, 17, 40][:B]
    full = torch.full((B, Tp + Tg), PAD_ID, dtype=torch.long, device=device)
    msk = torch.zeros(B, Tp + Tg, dtype=torch.bool, device=device)
    for b, p in enumerate(pads):
        full[b, p:] = ids[b, : Tp + Tg - p]; msk[b, p:] = True
    h_full, _ = mamba_seq(m, full, mask=msk)
    solo, _ = mamba_seq(m, full[B - 1:, pads[-1]:])
    d2 = (solo[0] - h_full[B - 1, pads[-1]:]).abs().max().item()
    print(f"(2) left-padded row vs unpadded row     max|diff| {d2:.2e}")
    assert d2 < _tol(solo), "left padding leaks into the real positions"

    h_p, st = mamba_seq(m, full[:, :Tp], mask=msk[:, :Tp], return_state=True)
    outs = [h_p[:, -1]]
    for t in range(Tp, Tp + Tg - 1):
        h_t, st = mamba_step(m, full[:, t], st)
        outs.append(h_t)
    d3 = (torch.stack(outs, 1) - h_full[:, Tp - 1:Tp + Tg - 1]).abs().max().item()
    print(f"(3) recurrent steps vs full forward     max|diff| {d3:.2e}")
    assert d3 < _tol(h_full), "recurrent decoding diverges from the full forward"

    if FAST["scan_kernel"]:
        FAST["scan_kernel"] = False
        pure, _ = mamba_seq(m, ids)
        FAST["scan_kernel"] = True
        d4 = (pure - new).abs().max().item()
        print(f"(4) fused kernel vs pure-PyTorch scan   max|diff| {d4:.2e}")
        if d4 > 5e-2 * max(1.0, pure.abs().max().item()):
            FAST["scan_kernel"] = False
            print("    kernel disagrees -> switched OFF for this session")
    with amp_ctx():
        h16, _ = mamba_seq(m, ids)
    cos = F.cosine_similarity(h16.float().reshape(-1, h16.size(-1)), new.reshape(-1, new.size(-1)), -1).min().item()
    print(f"(5) autocast vs fp32                    min cosine {cos:.5f}")
    print("Mamba self-test passed")


@torch.no_grad()
def check_peft_identity():
    """LoRA / LoRMA / adapters leave the model unchanged at init; QLoRA's 4-bit base stays close."""
    ids = torch.as_tensor(np.stack([d[:128] for d in tr_doc if len(d) >= 128][:2]), dtype=torch.long, device=device)
    ref, _ = mamba_seq(MAMBA_BASE.to(device).eval(), ids)
    for meth in ("lora", "lorma", "adapter", "qlora"):
        g = copy.deepcopy(MAMBA_BASE)
        apply_backbone_method(g, SimpleNamespace(**{**BASE_CFG, "method": meth}))
        out, _ = mamba_seq(g.to(device).eval(), ids)
        if meth == "qlora":
            c = F.cosine_similarity(out.reshape(-1, out.size(-1)), ref.reshape(-1, ref.size(-1)), -1).mean().item()
            print(f"  {meth:8s} 4-bit base: mean cosine to fp32 {c:.4f}")
        else:
            print(f"  {meth:8s} identity at init: max|diff| {(out - ref).abs().max().item():.2e}")
        del g
    torch.cuda.empty_cache()


check_mamba_patch()

(1) patched vs Phase-1 forward          max|diff| 0.00e+00
(2) left-padded row vs unpadded row     max|diff| 2.80e-06
(3) recurrent steps vs full forward     max|diff| 2.26e-06
(5) autocast vs fp32                    min cosine 0.99998
Mamba self-test passed


In [ ]:
# ============================================================
# 2. DOCUMENT MEMORY ("encoder" side of the Mamba summariser)
#
# The frozen Phase-1 Mamba reads every 512-token chunk once (its training length). Each chunk is compressed to
# m slot vectors = mean of the Mamba hidden states over m consecutive segments of the chunk. K chunks x m slots =
# the document memory, cached in fp16 on Drive. A small trainable bridge (bidirectional Transformer over the slots
# + projection into the token-embedding space) turns it into soft tokens; Mamba then reads them like ordinary
# input in one left-to-right pass and writes the summary:
#
#   [ pad .. pad | prompt? | mem_1 .. mem_N | <s> | y_1 .. y_L | </s> ]     (no window limit: Mamba is recurrent)
#                                             loss only on y_1 .. </s>
# ============================================================
@torch.no_grad()
def extract_memory(docs, K, m, strategy, layer=None, tok_budget=32768, max_bs=128, log_every=50):
    bb = MAMBA_BASE.to(device).eval()
    per_doc = [select_chunks(chunk_ids(d, PRETRAIN_LEN, PRETRAIN_LEN), K, strategy) for d in docs]
    flat = [c for cs in per_doc for c in cs]
    n, d = len(flat), bb.token_embedding.embedding_dim
    lens = np.fromiter((max(len(c), 1) for c in flat), dtype=np.int64, count=n)
    order = np.argsort(lens, kind="stable")
    out = torch.empty(n, m, d, dtype=torch.float16)
    print(f"  {n:,} chunks | {int(lens.sum()):,} tokens -> {m} slots per chunk")

    def run(sel):
        T = int(lens[sel].max())
        arr = np.full((len(sel), T), PAD_ID, dtype=np.int64)
        for j, i in enumerate(sel):
            c = flat[i]
            if len(c):
                arr[j, :len(c)] = c
        ids = torch.from_numpy(arr).to(device)
        ln = torch.from_numpy(lens[sel]).to(device)
        with amp_ctx():
            h, _ = mamba_seq(bb, ids, upto=layer)                               # right padding is safe (causal)
        h = h.float()
        pos = torch.arange(T, device=device)[None, :]
        seg = (pos * m // ln[:, None]).clamp(max=m - 1)                         # segment id of every position
        oh = F.one_hot(seg, m).float() * (pos < ln[:, None]).unsqueeze(-1).float()   # [b,T,m]
        slots = torch.einsum("btm,btd->bmd", oh, h) / oh.sum(1).clamp_min(1).unsqueeze(-1)
        out[torch.from_numpy(sel)] = slots.half().cpu()

    def safe_run(sel):
        try:
            run(sel)
        except torch.cuda.OutOfMemoryError:
            if len(sel) == 1:
                raise
            torch.cuda.empty_cache(); safe_run(sel[: len(sel) // 2]); safe_run(sel[len(sel) // 2:])

    t0, s, nb = time.time(), 0, 0
    while s < n:
        e = s + 1
        while e < n and (e - s) < max_bs and (e + 1 - s) * lens[order[e]] <= tok_budget:
            e += 1
        safe_run(order[s:e]); s = e; nb += 1
        if nb % log_every == 0 or s >= n:
            print(f"    {s:,}/{n:,} chunks ({time.time() - t0:.0f}s)")
    feats, p = [], 0
    for cs in per_doc:
        feats.append(out[p:p + len(cs)].reshape(-1, d)); p += len(cs)        # [n_chunks*m, d]
    return feats


def get_memory(split, docs, cfg):
    name = f"mem_{split}_s{SPLIT_SEED}_K{cfg.mem_chunks}_m{cfg.mem_slots}_{cfg.chunk_strategy}_L{cfg.mem_layer}.pt"
    dp, lp = os.path.join(p6_dir, name), os.path.join("/content/feat_cache", name)
    os.makedirs("/content/feat_cache", exist_ok=True)
    if not os.path.exists(lp) and os.path.exists(dp):
        shutil.copy(dp, lp)
    if os.path.exists(lp):
        return torch.load(lp)
    print(f"extracting Mamba memory for '{split}' ...")
    feats = extract_memory(docs, cfg.mem_chunks, cfg.mem_slots, cfg.chunk_strategy,
                           None if cfg.mem_layer < 0 else cfg.mem_layer)
    torch.save(feats, lp); shutil.copy(lp, dp)
    return feats


class MemoryBridge(nn.Module):
    """slots [B,N,d] (left-padded) -> soft tokens in Mamba's input-embedding space.
    Adds chunk / slot position embeddings, mixes the slots bidirectionally (the causal Mamba features of one chunk
    never saw the other chunks), then projects and rescales to the token-embedding norm."""

    def __init__(self, d, n_chunks, n_slots, emb_std, layers=1, heads=8, dropout=0.1):
        super().__init__()
        self.n_slots = n_slots
        self.chunk_pos, self.slot_pos = nn.Embedding(n_chunks, d), nn.Embedding(n_slots, d)
        nn.init.normal_(self.chunk_pos.weight, std=0.02); nn.init.normal_(self.slot_pos.weight, std=0.02)
        self.ln_in = nn.LayerNorm(d)
        self.mix = (nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, activation="gelu",
                                                                     batch_first=True, norm_first=True),
                                          layers, enable_nested_tensor=False) if layers > 0 else None)
        self.ln_out, self.proj = nn.LayerNorm(d), nn.Linear(d, d)
        self.gain = nn.Parameter(torch.tensor(float(emb_std)))

    def forward(self, feats, mask, slot_idx):                 # slot_idx [B,N] = index of the slot inside its doc
        z = self.ln_in(feats) + self.chunk_pos(slot_idx // self.n_slots) + self.slot_pos(slot_idx % self.n_slots)
        if self.mix is not None:
            z = self.mix(z, src_key_padding_mask=~mask)
        return self.proj(self.ln_out(z)) * self.gain

In [ ]:
# ============================================================
# 3. MAMBA SUMMARISER + ADAPTATION METHODS
#
# Targets: in_proj, x_proj, out_proj of every block. dt_proj / conv1d / A_log / D stay frozen (the fused scan kernel
# reads dt_proj.weight directly). Prefix tuning does not apply (Mamba has no attention keys/values to prefix);
# prompt tuning (soft input tokens) is its natural counterpart and is included.
# ============================================================
MAMBA_LINEARS = ("in_proj", "x_proj", "out_proj")


def _is_target(name):
    return name.startswith("layers.") and name.split(".")[-1] in MAMBA_LINEARS


def apply_backbone_method(m, cfg):
    meth = cfg.method
    freeze(m)
    d = m.token_embedding.embedding_dim
    if meth in ("full", "kd"):
        for p in m.parameters():
            p.requires_grad = True
        return "all Mamba weights"
    if meth == "partial":
        k = cfg.unfreeze_top_k
        for blk in m.layers[len(m.layers) - k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in m.norm_final.parameters():
            p.requires_grad = True
        return f"top {k} blocks + final norm"
    if meth == "frozen":
        return "backbone frozen (memory bridge only)"
    if meth == "lora":
        n = replace_linears(m, lambda l: LoRALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRA r={cfg.r} on {len(n)} linears"
    if meth == "lorma":
        n = replace_linears(m, lambda l: LoRMALinear(l, cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"LoRMA+ r={cfg.r} on {len(n)} linears"
    if meth == "qlora":
        n = replace_linears(m, lambda l: LoRALinear(NF4Linear(l), cfg.r, cfg.alpha, cfg.lora_dropout), _is_target)
        return f"QLoRA (NF4 base) r={cfg.r} on {len(n)} linears"
    if meth == "adapter":
        for blk in m.layers:
            blk.adapter = Adapter(d, cfg.adapter_dim, cfg.adapter_dropout)    # after the mixer, before the residual
        return f"bottleneck adapters ({cfg.adapter_dim}) after every mixer"
    if meth == "prompt":
        return f"prompt tuning, {cfg.prompt_len} soft tokens"
    raise ValueError(f"unknown method {meth!r} (prefix tuning needs attention; Mamba has none -> use 'prompt')")


def left_align(x, mask):
    """move every row's real positions to the END (padding first), keeping their order: x [B,L,d], mask [B,L]"""
    order = torch.sort(mask.int(), dim=1, stable=True).indices                 # False (pad) first, then True
    return x.gather(1, order[:, :, None].expand(-1, -1, x.size(-1))), mask.gather(1, order)


class MambaSummarizer(nn.Module):
    def __init__(self, mamba, cfg):
        super().__init__()
        self.g, self.cfg = mamba, cfg
        self.pretrained_param_ids = {id(p) for p in mamba.parameters()}
        self.method_desc = apply_backbone_method(mamba, cfg)
        d = mamba.token_embedding.embedding_dim
        emb_std = float(mamba.token_embedding.weight.detach().float().std())
        self.bridge = (MemoryBridge(d, cfg.mem_chunks, cfg.mem_slots, emb_std, cfg.bridge_layers)
                       if cfg.mem_mode == "memory" else None)
        self.prompt = None
        if cfg.method == "prompt":
            init = mamba.token_embedding.weight.detach()[torch.randint(1000, 20000, (cfg.prompt_len,))].clone()
            self.prompt = nn.Parameter(init)

    # ---- prefix: [pad .. pad | prompt | memory or document text | <s>] ------------------------
    def _prefix_inputs(self, b):
        emb = self.g.token_embedding
        if self.bridge is not None:
            mask = b["mem_mask"].to(device)
            src = self.bridge(b["mem"].to(device).float(), mask, b["slot_idx"].to(device))
        else:
            mask = b["text_mask"].to(device)
            src = emb(b["text"].to(device))
        B, dt = src.size(0), emb.weight.dtype
        parts, masks = [src.to(dt)], [mask]
        if self.prompt is not None:
            parts.insert(0, self.prompt.to(dt).unsqueeze(0).expand(B, -1, -1))
            masks.insert(0, mask.new_ones(B, self.prompt.size(0)))
        x, m = left_align(torch.cat(parts, 1), torch.cat(masks, 1))            # the recurrence must not cross padding
        bos = emb(torch.full((B, 1), BOS_ID, dtype=torch.long, device=device))
        return torch.cat([x, bos.to(x.dtype)], 1), torch.cat([m, m.new_ones(B, 1)], 1)

    def _summary_states(self, b):
        pe, pm = self._prefix_inputs(b)
        y, ym, tgt = b["y"].to(device), b["y_mask"].to(device), b["tgt"].to(device)
        x = torch.cat([pe, self.g.token_embedding(y).to(pe.dtype)], 1)
        h, _ = mamba_seq(self.g, embeds=x, mask=torch.cat([pm, ym], 1), ckpt=self.cfg.grad_ckpt)
        Tp, L = pe.size(1), y.size(1)
        hs = h[:, Tp - 1:Tp + L]                                  # <s> predicts y_1 ... last real y predicts </s>
        sel = tgt != -100
        return hs[sel], tgt[sel]

    def logits_targets(self, b):
        h, t = self._summary_states(b)
        return self.g.lm_head(h), t

    def loss(self, b, teacher=None):
        logits, t = self.logits_targets(b)
        ce = F.cross_entropy(logits.float(), t, label_smoothing=self.cfg.label_smoothing)
        if teacher is None:
            return ce
        with torch.no_grad():
            tl, tt = teacher.logits_targets(b)
        assert tt.shape == t.shape
        T = self.cfg.kd_temp
        kl = F.kl_div(F.log_softmax(logits.float() / T, -1), F.log_softmax(tl.float() / T, -1),
                      log_target=True, reduction="batchmean") * (T * T)
        return (1 - self.cfg.kd_alpha) * ce + self.cfg.kd_alpha * kl

    @torch.no_grad()
    def generate(self, b, gen):
        g = self.g
        pe, pm = self._prefix_inputs(b)

        def start():
            h, st = mamba_seq(g, embeds=pe, mask=pm, return_state=True)
            return st, g.lm_head(h[:, -1]).float()

        def step(st, tok):
            h, st = mamba_step(g, tok, st)
            return st, g.lm_head(h).float()

        return beam_search(start, step, reorder_states, num_beams=gen["num_beams"], max_new=gen["max_new"],
                           min_new=gen["min_new"], len_penalty=gen["len_penalty"],
                           no_repeat_ngram=gen["no_repeat_ngram"], rep_penalty=gen["rep_penalty"],
                           banned=(PAD_ID, BOS_ID))


def build_student_backbone(teacher_m, n_layers):
    """Smaller Mamba initialised from evenly spaced blocks of the fine-tuned teacher (shrink-and-fine-tune)."""
    s = MambaLM(vocab_size=len(tokenizer.itos), pad_token_id=PAD_ID, **{**MAMBA_CFG, "n_layers": n_layers})
    keep = np.linspace(0, len(teacher_m.layers) - 1, n_layers).round().astype(int).tolist()
    s.token_embedding.load_state_dict(teacher_m.token_embedding.state_dict())
    s.norm_final.load_state_dict(teacher_m.norm_final.state_dict())
    s.lm_head.load_state_dict(teacher_m.lm_head.state_dict())
    for j, i in enumerate(keep):
        s.layers[j].load_state_dict(teacher_m.layers[i].state_dict())
    print(f"student: {n_layers} blocks from teacher blocks {keep} | {sum(p.numel() for p in s.parameters())/1e6:.1f}M "
          f"params (teacher {sum(p.numel() for p in teacher_m.parameters())/1e6:.1f}M)")
    return s


INPUT_KEYS = ("mem_mode", "mem_chunks", "mem_slots", "chunk_strategy", "mem_layer", "bridge_layers",
              "max_sum_tok", "text_tok")


def build_summarizer(cfg, backbone=None):
    g = copy.deepcopy(MAMBA_BASE) if backbone is None else backbone
    return MambaSummarizer(g, cfg).to(device)


def make_student_summarizer(teacher, cfg):
    sm = build_summarizer(cfg, backbone=build_student_backbone(teacher.g, cfg.student_layers))
    if teacher.bridge is not None:
        sm.bridge.load_state_dict(teacher.bridge.state_dict())
    return sm

In [ ]:
# ============================================================
# 4. DATASETS / COLLATE
#   mem_mode="memory": cached Mamba slot vectors (left-padded)
#   mem_mode="text"  : the selected document chunks as plain tokens, up to text_tok (left-padded). Mamba has no
#                      window, so this can be much longer than 512 (it only saw 512 in pre-training, though).
# ============================================================
SPLITS = {"train": (train_recs, tr_doc, tr_sum), "val": (val_recs, va_doc, va_sum), "test": (test_recs, te_doc, None)}
_DS = {}


def doc_text_tokens(d, cfg):
    ch = select_chunks(chunk_ids(d, PRETRAIN_LEN, PRETRAIN_LEN), max(1, math.ceil(cfg.text_tok / PRETRAIN_LEN)),
                       cfg.chunk_strategy)
    return np.concatenate([np.asarray(c, dtype=np.int64) for c in ch])[:cfg.text_tok]


class MambaSummDS(Dataset):
    def __init__(self, split, cfg):
        recs, docs, sums = SPLITS[split]
        self.ids = [r["id"] for r in recs]
        self.src = (get_memory(split, docs, cfg) if cfg.mem_mode == "memory"
                    else [doc_text_tokens(d, cfg) for d in docs])
        self.sums = sums

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        return self.ids[i], self.src[i], (None if self.sums is None else self.sums[i])


def make_collate(cfg, with_target):
    def collate(batch):
        B = len(batch)
        out = {"ids": [b[0] for b in batch]}
        if cfg.mem_mode == "memory":
            N = max(b[1].size(0) for b in batch); d = batch[0][1].size(1)
            mem = torch.zeros(B, N, d, dtype=torch.float16)
            mask = torch.zeros(B, N, dtype=torch.bool); slot = torch.zeros(B, N, dtype=torch.long)
            for i, (_, f, _) in enumerate(batch):
                n = f.size(0)
                mem[i, N - n:] = f; mask[i, N - n:] = True; slot[i, N - n:] = torch.arange(n)
            out.update(mem=mem, mem_mask=mask, slot_idx=slot)
        else:
            N = max(max(len(b[1]), 1) for b in batch)
            txt = torch.full((B, N), PAD_ID, dtype=torch.long); mask = torch.zeros(B, N, dtype=torch.bool)
            for i, (_, t, _) in enumerate(batch):
                if len(t):
                    txt[i, N - len(t):] = torch.from_numpy(t); mask[i, N - len(t):] = True
            out.update(text=txt, text_mask=mask)
        if with_target:
            ys = [np.asarray(b[2][:cfg.max_sum_tok], dtype=np.int64) for b in batch]
            L = max(max(len(y) for y in ys), 1)
            y = torch.full((B, L), PAD_ID, dtype=torch.long); ym = torch.zeros(B, L, dtype=torch.bool)
            tgt = torch.full((B, L + 1), -100, dtype=torch.long)
            for i, s in enumerate(ys):
                n = len(s)
                y[i, :n] = torch.from_numpy(s); ym[i, :n] = True
                tgt[i, :n] = torch.from_numpy(s); tgt[i, n] = EOS_ID
            out.update(y=y, y_mask=ym, tgt=tgt)
        return out
    return collate


def make_loader(split, cfg, shuffle, gen=False, limit=None):
    key = (split, cfg.mem_mode, cfg.mem_chunks, cfg.mem_slots, cfg.chunk_strategy, cfg.mem_layer, cfg.text_tok)
    if key not in _DS:
        _DS[key] = MambaSummDS(split, cfg)
    ds = _DS[key]
    if limit:
        ds = torch.utils.data.Subset(ds, list(range(min(limit, len(ds)))))
    elif split == "train" and cfg.max_train_docs:
        ds = torch.utils.data.Subset(ds, list(range(min(cfg.max_train_docs, len(ds)))))
    has_y = split != "test" and not gen
    return DataLoader(ds, batch_size=cfg.gen_bs if gen else cfg.batch_size, shuffle=shuffle,
                      collate_fn=make_collate(cfg, has_y), num_workers=0, drop_last=False)

In [ ]:
# ============================================================
# 5. CONFIGURATION
# ============================================================
MAX_SUM_TOK = 768                 # Mamba has no window: the summary budget is set by the data, not the model

BASE_CFG = dict(
    method="lora",
    # document memory: K chunks x m slots (16 x 8 = 128 soft tokens) | or plain text of text_tok tokens
    mem_mode="memory", mem_chunks=16, mem_slots=8, chunk_strategy="uniform", mem_layer=-1, bridge_layers=1,
    text_tok=2048,
    max_sum_tok=None,             # filled in by make_cfg: min(p95 of reference lengths, MAX_SUM_TOK)
    # adaptation hyper-parameters
    r=16, alpha=32, lora_dropout=0.05, adapter_dim=64, adapter_dropout=0.1, prompt_len=20, unfreeze_top_k=4,
    # optimisation
    lr=5e-4, lr_backbone=5e-5, wd=0.01, epochs=6, patience=2, batch_size=4, accum=4, warmup=0.05, clip=1.0,
    label_smoothing=0.1, grad_ckpt=True, seed=42, log_every=100, max_train_docs=None,
    # model selection during training (generation on a slice of val, not loss)
    val_loss_batches=20, n_eval_gen=64, select_metric="rougeL", gen_bs=8, n_eval_final=None,
    # knowledge distillation (student: 10 of the 20 blocks, taken from the fine-tuned teacher)
    teacher_ckpt=None, student_layers=10, kd_alpha=0.5, kd_temp=2.0,
)

METHOD_DEFAULTS = {
    "full":    dict(lr=5e-4, lr_backbone=3e-5),
    "partial": dict(lr=5e-4, lr_backbone=5e-5),
    "frozen":  dict(lr=1e-3),
    "lora":    dict(lr=5e-4),
    "lorma":   dict(lr=5e-4),
    "qlora":   dict(lr=5e-4),
    "adapter": dict(lr=5e-4),
    "prompt":  dict(lr=3e-3),
    "kd":      dict(lr=5e-4, lr_backbone=5e-5),
}


def make_cfg(**over):
    meth = over.get("method", BASE_CFG["method"])
    c = SimpleNamespace(**{**BASE_CFG, **METHOD_DEFAULTS.get(meth, {}), **over})
    c.max_sum_tok = c.max_sum_tok or min(SUM_TOK_P95, MAX_SUM_TOK)
    return c


GEN = dict(num_beams=4, max_new=min(int(SUM_TOK_P90 * 1.1), MAX_SUM_TOK), min_new=SUM_TOK_P10, len_penalty=1.0,
           no_repeat_ngram=3, rep_penalty=1.0, max_words=None)

_c = make_cfg()
print(f"memory: {_c.mem_chunks} chunks x {_c.mem_slots} slots = {_c.mem_chunks * _c.mem_slots} soft tokens | "
      f"summary budget {_c.max_sum_tok} tokens (reference p95 {SUM_TOK_P95}) | "
      f"generation {GEN['min_new']}-{GEN['max_new']} tokens, {GEN['num_beams']} beams")
if _c.mem_chunks * _c.mem_slots + _c.max_sum_tok > PRETRAIN_LEN:
    print(f"NOTE: training sequences (~{_c.mem_chunks * _c.mem_slots + 1 + _c.max_sum_tok} positions) are longer than "
          f"the {PRETRAIN_LEN} tokens Mamba was pre-trained on. Mamba is recurrent so this runs fine; fine-tuning "
          f"teaches it the longer horizon.")

check_peft_identity()

memory: 16 chunks x 8 slots = 128 soft tokens | summary budget 768 tokens (reference p95 2399) | generation 342-768 tokens, 4 beams
NOTE: training sequences (~897 positions) are longer than the 512 tokens Mamba was pre-trained on. Mamba is recurrent so this runs fine; fine-tuning teaches it the longer horizon.
  lora     identity at init: max|diff| 2.15e-06
  lorma    identity at init: max|diff| 2.15e-06
  adapter  identity at init: max|diff| 2.15e-06
  qlora    4-bit base: mean cosine to fp32 0.9964


In [ ]:
# ============================================================
# 6. EXPERIMENT RUNNER
# ============================================================
RESULTS = {}
_res_path = os.path.join(p6_dir, "results.json")
if os.path.exists(_res_path):
    RESULTS = json.load(open(_res_path))


def _teacher_from_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    tcfg = SimpleNamespace(**ck["cfg"])
    t = build_summarizer(tcfg)
    load_trainable(t, path)
    return t, tcfg


def evaluate_and_submit(sm, cfg, name, submit=True):
    t0 = time.time()
    val_pred = generate_split(sm, "val", cfg, n_docs=getattr(cfg, "n_eval_final", None))
    m = score_split(val_pred, val_recs, bert=True)
    print(f"[{name}] VAL ({len(val_pred)} docs): {fmt_metrics(m)} ({time.time()-t0:.0f}s)")
    RESULTS[name] = dict(val=m, method=cfg.method, desc=sm.method_desc,
                         trainable=n_params(sm), total=n_params(sm, trainable=False),
                         cfg={k: v for k, v in vars(cfg).items() if isinstance(v, (int, float, str, bool, type(None)))})
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    json.dump(val_pred, open(os.path.join(p6_dir, f"valpred_{name}.json"), "w"), ensure_ascii=False)
    if submit:
        test_pred = generate_split(sm, "test", cfg)
        write_submission(test_pred, name)
    return m


def run_experiment(name, submit=True, **over):
    cfg = make_cfg(**over)
    set_seed(cfg.seed)
    teacher = None
    if cfg.method == "kd":
        assert cfg.teacher_ckpt and os.path.exists(cfg.teacher_ckpt), "kd needs teacher_ckpt (e.g. the full fine-tune)"
        teacher, tcfg = _teacher_from_ckpt(cfg.teacher_ckpt)
        for k in INPUT_KEYS:
            setattr(cfg, k, getattr(tcfg, k))                       # student reads exactly the teacher's inputs
        sm = make_student_summarizer(teacher, cfg)
    else:
        sm = build_summarizer(cfg)
    print(f"\n=== {name} | {sm.method_desc} | trainable {n_params(sm)/1e6:.2f}M / {n_params(sm, False)/1e6:.1f}M ===")
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    train_summarizer(sm, cfg, name, teacher=teacher)
    train_min = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0
    del teacher; gc.collect(); torch.cuda.empty_cache()
    m = evaluate_and_submit(sm, cfg, name, submit)
    RESULTS[name].update(train_min=train_min, peak_gb=peak)
    json.dump(RESULTS, open(_res_path, "w"), indent=1)
    del sm; gc.collect(); torch.cuda.empty_cache()
    return m


def resubmit(name, **gen_over):
    """Re-generate val + test from a saved checkpoint with different decoding (no retraining)."""
    path = os.path.join(p6_dir, f"ckpt_{name}.pt")
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = SimpleNamespace(**ck["cfg"])
    if cfg.method == "kd":
        teacher, _ = _teacher_from_ckpt(cfg.teacher_ckpt)
        sm = make_student_summarizer(teacher, cfg); del teacher
    else:
        sm = build_summarizer(cfg)
    load_trainable(sm, path)
    saved = dict(GEN); GEN.update(gen_over)
    tag = name + "_" + "_".join(f"{k}{v}" for k, v in gen_over.items())
    try:
        return evaluate_and_submit(sm, cfg, tag)
    finally:
        GEN.clear(); GEN.update(saved); del sm; gc.collect(); torch.cuda.empty_cache()


def show_results():
    print(f"\n{'experiment':26s} {'method':9s} {'BERTScore':>9s} {'R1':>7s} {'R2':>7s} {'RL':>7s} {'words':>6s} "
          f"{'train M':>8s} {'total M':>8s} {'min':>6s} {'GB':>5s}")
    for n, r in RESULTS.items():
        v = r["val"]; bs = v.get("bertscore")
        print(f"{n:26s} {r['method']:9s} {('%.4f' % bs) if bs is not None else 'n/a':>9s} {v['rouge1']:7.4f} "
              f"{v['rouge2']:7.4f} {v['rougeL']:7.4f} {v['avg_words']:6.0f} {r['trainable']/1e6:8.2f} "
              f"{r['total']/1e6:8.1f} {r.get('train_min', 0):6.0f} {r.get('peak_gb', 0):5.1f}")

In [ ]:
# ============================================================
# 7. RUN
# ============================================================
# (0) Smoke test FIRST (a few minutes): whole pipeline on 64 documents, no submission.
#     Extracts + caches the Mamba memory for train/val/test on first use (one-off).
run_experiment("M0_smoke", method="lora", max_train_docs=64, epochs=1, n_eval_gen=8, n_eval_final=16, submit=False)

# (1) Six substantially different adaptation methods (LoRMA compulsory, KD mandatory). Same memory input and
#     decoding, so differences come from the adaptation method.
# run_experiment("M1_full",    method="full", batch_size=2, accum=8)        # also the KD teacher
# run_experiment("M2_lora",    method="lora")
# run_experiment("M3_lorma",   method="lorma")                             # compulsory
# run_experiment("M4_adapter", method="adapter")
# run_experiment("M5_prompt",  method="prompt")
# run_experiment("M6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_M1_full.pt"), student_layers=10)
#   (the KD teacher must be a 'full' or 'partial' run: the student copies its plain block weights)

# (2) Optional extras / ablations
# run_experiment("M7_qlora",   method="qlora")                             # 4-bit base: memory vs quality
# run_experiment("M8_partial", method="partial", unfreeze_top_k=4)
# run_experiment("M9_frozen",  method="frozen")                            # bridge only
# run_experiment("M10_text2k", method="lora", mem_mode="text", text_tok=2048, batch_size=2, accum=8)
#   ^ Mamba-specific: read 2048 raw document tokens directly (no window limit) instead of compressed memory
# run_experiment("M11_head",   method="lora", chunk_strategy="head_tail")

# (3) Decoding sweeps on a trained checkpoint (no retraining):
# resubmit("M3_lorma", len_penalty=1.2)
# resubmit("M3_lorma", num_beams=1)

show_results()


=== M0_smoke | LoRA r=16 on 60 linears | trainable 5.75M / 92.0M ===
extracting Mamba memory for 'train' ...
  57,269 chunks | 27,714,602 tokens -> 8 slots per chunk
    5,506/57,269 chunks (138s)
    8,765/57,269 chunks (310s)
    11,965/57,269 chunks (482s)
    15,165/57,269 chunks (654s)
    18,365/57,269 chunks (826s)
    21,565/57,269 chunks (998s)
    24,765/57,269 chunks (1170s)
    27,965/57,269 chunks (1342s)
    31,165/57,269 chunks (1514s)
    34,365/57,269 chunks (1686s)
    37,565/57,269 chunks (1858s)
    40,765/57,269 chunks (2030s)
    43,965/57,269 chunks (2202s)
    47,165/57,269 chunks (2374s)
    50,365/57,269 chunks (2546s)
    53,565/57,269 chunks (2718s)
    56,765/57,269 chunks (2890s)
    57,269/57,269 chunks (2917s)
extracting Mamba memory for 'val' ...
  3,024 chunks | 1,465,791 tokens -> 8 slots per chunk
    3,024/3,024 chunks (157s)
[M0_smoke] 16 batches/epoch x 1 epochs | accum 4 | trainable new 5.75M + backbone 0.00M


/tmp/ipykernel_524/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run += float(loss)


[M0_smoke] ep 01 train 5.1486 | val loss 5.0944 | BERTScore n/a | R1 0.3915 R2 0.1179 RL 0.1903 | words 663 (ref 501) (67s)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[M0_smoke] VAL (16 docs): BERTScore 0.8097 | R1 0.3929 R2 0.1165 RL 0.1934 | words 664 (ref 521) (80s)

experiment                 method    BERTScore      R1      R2      RL  words  train M  total M    min    GB
M0_smoke                   lora         0.8097  0.3929  0.1165  0.1934    664     5.75     92.0     52  20.7


In [ ]:
run_experiment("M1_full",    method="full", batch_size=2, accum=8)        # also the KD teacher
run_experiment("M2_lora",    method="lora")
run_experiment("M3_lorma",   method="lorma")                             # compulsory
run_experiment("M4_adapter", method="adapter")
run_experiment("M5_prompt",  method="prompt")
run_experiment("M6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_M1_full.pt"), student_layers=10)
  # (the KD teacher must be a 'full' or 'partial' run: the student copies its plain block weights)


=== M1_full | all Mamba weights | trainable 89.68M / 89.7M ===
[M1_full] 3339 batches/epoch x 6 epochs | accum 8 | trainable new 3.43M + backbone 86.25M
[M1_full] ep 1 batch 100/3339 loss 5.2845 (106s)
[M1_full] ep 1 batch 200/3339 loss 5.2134 (212s)
[M1_full] ep 1 batch 300/3339 loss 5.1561 (318s)
[M1_full] ep 1 batch 400/3339 loss 5.1285 (426s)
[M1_full] ep 1 batch 500/3339 loss 5.1022 (535s)
[M1_full] ep 1 batch 600/3339 loss 5.0801 (641s)
[M1_full] ep 1 batch 700/3339 loss 5.0563 (749s)
[M1_full] ep 1 batch 800/3339 loss 5.0369 (854s)
[M1_full] ep 1 batch 900/3339 loss 5.0266 (960s)
[M1_full] ep 1 batch 1000/3339 loss 5.0121 (1066s)
[M1_full] ep 1 batch 1100/3339 loss 5.0001 (1170s)
[M1_full] ep 1 batch 1200/3339 loss 4.9886 (1276s)
[M1_full] ep 1 batch 1300/3339 loss 4.9772 (1381s)
[M1_full] ep 1 batch 1400/3339 loss 4.9682 (1488s)
[M1_full] ep 1 batch 1500/3339 loss 4.9585 (1596s)
[M1_full] ep 1 batch 1600/3339 loss 4.9524 (1701s)
[M1_full] ep 1 batch 1700/3339 loss 4.9435 (1810

In [ ]:
run_experiment("M5_prompt",  method="prompt")


=== M5_prompt | prompt tuning, 20 soft tokens | trainable 3.44M / 89.7M ===
[M5_prompt] 1670 batches/epoch x 6 epochs | accum 4 | trainable new 3.44M + backbone 0.00M


/tmp/ipykernel_825/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run += float(loss)


[M5_prompt] ep 1 batch 100/1670 loss 5.2122 (192s)
[M5_prompt] ep 1 batch 200/1670 loss 5.1990 (382s)
[M5_prompt] ep 1 batch 300/1670 loss 5.2100 (573s)
[M5_prompt] ep 1 batch 400/1670 loss 5.1985 (763s)
[M5_prompt] ep 1 batch 500/1670 loss 5.1944 (954s)
[M5_prompt] ep 1 batch 600/1670 loss 5.1948 (1145s)
[M5_prompt] ep 1 batch 700/1670 loss 5.1957 (1336s)
[M5_prompt] ep 1 batch 800/1670 loss 5.1966 (1524s)
[M5_prompt] ep 1 batch 900/1670 loss 5.1932 (1717s)
[M5_prompt] ep 1 batch 1000/1670 loss 5.1925 (1910s)
[M5_prompt] ep 1 batch 1100/1670 loss 5.1904 (2099s)
[M5_prompt] ep 1 batch 1200/1670 loss 5.1906 (2291s)
[M5_prompt] ep 1 batch 1300/1670 loss 5.1882 (2479s)
[M5_prompt] ep 1 batch 1400/1670 loss 5.1887 (2672s)
[M5_prompt] ep 1 batch 1500/1670 loss 5.1881 (2863s)
[M5_prompt] ep 1 batch 1600/1670 loss 5.1883 (3052s)
[M5_prompt] ep 01 train 5.1888 | val loss 5.1272 | BERTScore n/a | R1 0.3665 R2 0.1159 RL 0.1929 | words 666 (ref 620) (3354s)
[M5_prompt] ep 2 batch 100/1670 loss 5.

In [ ]:
run_experiment("M6_kd",      method="kd", teacher_ckpt=os.path.join(p6_dir, "ckpt_M1_full.pt"), student_layers=10)

student: 10 blocks from teacher blocks [0, 2, 4, 6, 8, 11, 13, 15, 17, 19] | 59.9M params (teacher 86.2M)

=== M6_kd | all Mamba weights | trainable 63.33M / 63.3M ===
[M6_kd] 1670 batches/epoch x 6 epochs | accum 4 | trainable new 3.43M + backbone 59.90M


/tmp/ipykernel_1611/1471079892.py:100: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run += float(loss)


[M6_kd] ep 1 batch 100/1670 loss 2.6011 (625s)
[M6_kd] ep 1 batch 200/1670 loss 2.5331 (1248s)
[M6_kd] ep 1 batch 300/1670 loss 2.4996 (1877s)
[M6_kd] ep 1 batch 400/1670 loss 2.4740 (2505s)
[M6_kd] ep 1 batch 500/1670 loss 2.4590 (3130s)
[M6_kd] ep 1 batch 600/1670 loss 2.4430 (3762s)
[M6_kd] ep 1 batch 700/1670 loss 2.4306 (4390s)
[M6_kd] ep 1 batch 800/1670 loss 2.4223 (5018s)
[M6_kd] ep 1 batch 900/1670 loss 2.4147 (5643s)
[M6_kd] ep 1 batch 1000/1670 loss 2.4082 (6266s)
[M6_kd] ep 1 batch 1100/1670 loss 2.4034 (6895s)
[M6_kd] ep 1 batch 1200/1670 loss 2.3992 (7521s)
[M6_kd] ep 1 batch 1300/1670 loss 2.3948 (8150s)
[M6_kd] ep 1 batch 1400/1670 loss 2.3919 (8780s)
[M6_kd] ep 1 batch 1500/1670 loss 2.3883 (9406s)
[M6_kd] ep 1 batch 1600/1670 loss 2.3844 (10024s)
[M6_kd] ep 01 train 2.3824 | val loss 4.5442 | BERTScore n/a | R1 0.4057 R2 0.1432 RL 0.2102 | words 421 (ref 620) (10567s)
[M6_kd] ep 2 batch 100/1670 loss 2.2969 (632s)
[M6_kd] ep 2 batch 200/1670 loss 2.2925 (1260s)
[M6_kd